# LightGBM Feature Engineering — V2-A

**Goal**

Test whether simple leakage-safe transaction relationship features improve the
LightGBM sanity model.

The experiment keeps fixed:

- LightGBM model configuration
- three frozen chronological folds
- preprocessing strategy
- class weighting
- evaluation metrics

Only the feature set changes.

**Baseline LightGBM feature set**

The current LightGBM sanity model achieved:

| Fold | Average Precision | PR Lift | Precision @ >=80% Recall | FP @ >=80% Recall | Alert Rate |
|---|---:|---:|---:|---:|---:|
| Fold 1 | 0.063079 | 32.15x | 2.277% | 13,991 | 6.90% |
| Fold 2 | 0.032902 | 33.72x | 0.567% | 66,063 | 13.77% |
| Fold 3 | 0.047368 | 44.46x | 1.573% | 51,490 | 5.42% |

The purpose of V2-A is to determine whether explicit transaction relationships
improve ranking and reduce false-positive burden before adding behavioral
history features.

# Imports


In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

In [3]:
PROJECT_ROOT = Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from walkforward import (
    FROZEN_FOLDS,
    build_walkforward_splits,
)

from model_evaluation import (
    evaluate_fold_scores,
    build_fold_metrics_table,
    summarize_temporal_stability,
)

# Load Sept 1-7 Development data

In [4]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "training_data.parquet"

In [5]:
dev_df = pd.read_parquet(DATA_PATH)
dev_df

,transaction_id,transaction_timestamp,from_bank_id,from_account_id,to_bank_id,to_account_id,amount_received,receiving_currency,amount_paid,payment_currency,payment_format,is_laundering,transaction_date,hour_of_day,day_of_week,is_weekend,same_currency_flag,same_bank_flag,log_amount_received,log_amount_paid
0,txn_000c8cd19174356a_000,2022-09-01 00:00:00,143430,810211AF0,143430,810211AF0,310038.60,Shekel,310038.60,Shekel,Reinvestment,0,2022-09-01,0,3,0,1,1,12.644455,12.644455
1,txn_0015c45f9cb45292_000,2022-09-01 00:00:00,123102,80D4F5640,123102,80D4F5640,16570.99,US Dollar,16570.99,US Dollar,Reinvestment,0,2022-09-01,0,3,0,1,1,9.715469,9.715469
2,txn_001dfc493364351d_000,2022-09-01 00:00:00,28629,805315930,221032,80E0BE6E0,2739.94,Euro,2739.94,Euro,Cheque,0,2022-09-01,0,3,0,1,0,7.916056,7.916056
3,txn_0024335c600d3870_000,2022-09-01 00:00:00,116,80E449450,116,80E449450,3459884.99,Swiss Franc,3459884.99,Swiss Franc,Reinvestment,0,2022-09-01,0,3,0,1,1,15.056746,15.056746
4,txn_00346c68f75d61f6_000,2022-09-01 00:00:00,13145,8102B1090,13145,8102B1090,348.74,US Dollar,348.74,US Dollar,Reinvestment,0,2022-09-01,0,3,0,1,1,5.857190,5.857190
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3731667,txn_fe07fc18f5b90ac6_000,2022-09-07 23:59:00,17615,806C1DED0,16388,806C1E370,2804.22,Euro,2804.22,Euro,ACH,0,2022-09-07,23,2,0,1,0,7.939237,7.939237
3731668,txn_fe2d139b8376e0ad_000,2022-09-07 23:59:00,213,80C18D670,234331,80C775110,2934716.59,Mexican Peso,2934716.59,Mexican Peso,Wire,0,2022-09-07,23,2,0,1,0,14.892122,14.892122
3731669,txn_fe85cd6131c1c550_000,2022-09-07 23:59:00,11657,8009D1ED0,121415,80C5F1370,84.82,Euro,84.82,Euro,Cheque,0,2022-09-07,23,2,0,1,0,4.452252,4.452252
3731670,txn_ff0799171be6367a_000,2022-09-07 23:59:00,2385,8016716A0,11157,801C44160,32.21,US Dollar,32.21,US Dollar,Credit Card,0,2022-09-07,23,2,0,1,0,3.502851,3.502851


In [6]:
dev_df["transaction_timestamp"] = pd.to_datetime(dev_df["transaction_timestamp"])

dev_df = dev_df.sort_values(
    "transaction_timestamp",
    kind="stable",
).reset_index(drop=True)

print(dev_df.shape)

print(
    dev_df["transaction_timestamp"].min(),
    "→",
    dev_df["transaction_timestamp"].max(),
)

(3731672, 20)
2022-09-01 00:00:00 → 2022-09-07 23:59:00


In [7]:
TEST_START = pd.Timestamp("2022-09-08")

assert (dev_df["transaction_timestamp"] < TEST_START).all(), (
    "September 8+ data detected!"
)

# Current V1 features

In [8]:
BASE_NUMERIC_FEATURES = [
    "amount_received",
    "amount_paid",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
    "log_amount_received",
    "log_amount_paid",
]

BASE_CATEGORICAL_FEATURES = [
    "receiving_currency",
    "payment_currency",
    "payment_format",
]

# V2-A features

**Feature 1 — currency_pair**: Instead of separately seeing USD/EUR. give the model:USD→EUR.The relationship itself can matter. \
**Feature 2 — payment_format_currency_pair (ACH | USD→USD)**: This explicitly represents payment mechanism + currency relationship which is the kind of interaction SGD could not naturally represent. \
**Feature 3 — payment_format_bank_relation**:expose the relationship directly. \
**Feature 4 — same_currency_log_amount_gap**: When payment and receiving currencies are the same ∣log(1+amount_paid)−log(1+amount_received)∣.If currencies differ, we leave it missing because directly comparing Yen with USD, for example, is not meaningful without FX context. 


In [9]:
def add_v2a_features(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add leakage-safe transaction-level relationship features.

    Every feature uses information available on the current
    transaction only.
    """

    result = df.copy()

    # ---------------------------------------------
    # Currency relationship
    # ---------------------------------------------

    result["currency_pair"] = (
        result["payment_currency"].astype(str)
        + "->"
        + result["receiving_currency"].astype(str)
    )

    # ---------------------------------------------
    # Payment format + currency relationship
    # ---------------------------------------------

    result["payment_format_currency_pair"] = (
        result["payment_format"].astype(str) + "|" + result["currency_pair"]
    )

    # ---------------------------------------------
    # Payment format + bank relationship
    # ---------------------------------------------

    bank_relation = np.where(
        result["same_bank_flag"] == 1,
        "same_bank",
        "cross_bank",
    )

    result["payment_format_bank_relation"] = (
        result["payment_format"].astype(str) + "|" + bank_relation
    )

    # ---------------------------------------------
    # Amount consistency when currency is the same
    # ---------------------------------------------

    same_currency = result["same_currency_flag"] == 1

    result["same_currency_log_amount_gap"] = np.nan

    result.loc[
        same_currency,
        "same_currency_log_amount_gap",
    ] = (
        result.loc[
            same_currency,
            "log_amount_paid",
        ]
        - result.loc[
            same_currency,
            "log_amount_received",
        ]
    ).abs()

    return result

In [10]:
v2a_df = add_v2a_features(dev_df)

In [11]:
V2A_NUMERIC_FEATURES = [
    "same_currency_log_amount_gap",
]

V2A_CATEGORICAL_FEATURES = [
    "currency_pair",
    "payment_format_currency_pair",
    "payment_format_bank_relation",
]

In [12]:
NUMERIC_FEATURES_V2A = BASE_NUMERIC_FEATURES + V2A_NUMERIC_FEATURES

CATEGORICAL_FEATURES_V2A = BASE_CATEGORICAL_FEATURES + V2A_CATEGORICAL_FEATURES

FEATURE_COLUMNS_V2A = NUMERIC_FEATURES_V2A + CATEGORICAL_FEATURES_V2A

In [13]:
v2a_df[
    [
        "payment_currency",
        "receiving_currency",
        "currency_pair",
        "payment_format",
        "payment_format_currency_pair",
        "same_bank_flag",
        "payment_format_bank_relation",
        "same_currency_flag",
        "amount_paid",
        "amount_received",
        "same_currency_log_amount_gap",
        "log_amount_paid",
        "log_amount_received",
    ]
].head(20)

,payment_currency,receiving_currency,currency_pair,payment_format,payment_format_currency_pair,same_bank_flag,payment_format_bank_relation,same_currency_flag,amount_paid,amount_received,same_currency_log_amount_gap,log_amount_paid,log_amount_received
0,Shekel,Shekel,Shekel->Shekel,Reinvestment,Reinvestment|Shekel->Shekel,1,Reinvestment|same_bank,1,3.100386e+05,3.100386e+05,0.0,12.644455,12.644455
1,US Dollar,US Dollar,US Dollar->US Dollar,Reinvestment,Reinvestment|US Dollar->US Dollar,1,Reinvestment|same_bank,1,1.657099e+04,1.657099e+04,0.0,9.715469,9.715469
2,Euro,Euro,Euro->Euro,Cheque,Cheque|Euro->Euro,0,Cheque|cross_bank,1,2.739940e+03,2.739940e+03,0.0,7.916056,7.916056
3,Swiss Franc,Swiss Franc,Swiss Franc->Swiss Franc,Reinvestment,Reinvestment|Swiss Franc->Swiss Franc,1,Reinvestment|same_bank,1,3.459885e+06,3.459885e+06,0.0,15.056746,15.056746
4,US Dollar,US Dollar,US Dollar->US Dollar,Reinvestment,Reinvestment|US Dollar->US Dollar,1,Reinvestment|same_bank,1,3.487400e+02,3.487400e+02,0.0,5.857190,5.857190
5,US Dollar,US Dollar,US Dollar->US Dollar,Reinvestment,Reinvestment|US Dollar->US Dollar,1,Reinvestment|same_bank,1,9.420000e+00,9.420000e+00,0.0,2.343727,2.343727
6,US Dollar,US Dollar,US Dollar->US Dollar,Reinvestment,Reinvestment|US Dollar->US Dollar,1,Reinvestment|same_bank,1,1.862060e+03,1.862060e+03,0.0,7.529976,7.529976
7,US Dollar,US Dollar,US Dollar->US Dollar,Credit Card,Credit Card|US Dollar->US Dollar,0,Credit Card|cross_bank,1,1.477701e+04,1.477701e+04,0.0,9.600896,9.600896
8,Shekel,Shekel,Shekel->Shekel,Credit Card,Credit Card|Shekel->Shekel,0,Credit Card|cross_bank,1,5.858236e+04,5.858236e+04,0.0,10.978206,10.978206
9,Euro,Euro,Euro->Euro,Reinvestment,Reinvestment|Euro->Euro,1,Reinvestment|same_bank,1,1.599000e+01,1.599000e+01,0.0,2.832625,2.832625


In [14]:
for col in V2A_CATEGORICAL_FEATURES:
    print(
        col,
        ":",
        v2a_df[col].nunique(),
        "unique values",
    )

currency_pair : 213 unique values
payment_format_currency_pair : 303 unique values
payment_format_bank_relation : 13 unique values


In [15]:
v2a_df["same_currency_log_amount_gap"].describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3680860.0
mean           0.0
std            0.0
min            0.0
50%            0.0
90%            0.0
95%            0.0
99%            0.0
99.9%          0.0
max            0.0
Name: same_currency_log_amount_gap, dtype: float64

In [16]:
print("Missing:", v2a_df["same_currency_log_amount_gap"].isna().sum())

print("Non-missing:", v2a_df["same_currency_log_amount_gap"].notna().sum())

Missing: 50812
Non-missing: 3680860


## Compare this feature for fraud vs legitimate

In [17]:
v2a_df.groupby("is_laundering")["same_currency_log_amount_gap"].describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99,
    ]
)

,count,mean,std,min,50%,90%,95%,99%,max
is_laundering,,,,,,,,,
0,3677833.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,3027.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Inspect the new categorical relationships

In [18]:
currency_pair_summary = v2a_df.groupby(
    "currency_pair",
    observed=True,
).agg(
    transactions=(
        "is_laundering",
        "size",
    ),
    frauds=(
        "is_laundering",
        "sum",
    ),
)

currency_pair_summary["fraud_rate"] = (
    currency_pair_summary["frauds"] / currency_pair_summary["transactions"]
)

currency_pair_summary.sort_values(
    "transactions",
    ascending=False,
).head(20)

,transactions,frauds,fraud_rate
currency_pair,,,
US Dollar->US Dollar,1364399,1111,0.000814
Euro->Euro,849027,778,0.000916
Swiss Franc->Swiss Franc,172787,120,0.000694
Yuan->Yuan,149763,124,0.000828
Shekel->Shekel,141074,69,0.000489
Rupee->Rupee,139194,114,0.000819
UK Pound->UK Pound,130629,91,0.000697
Ruble->Ruble,113611,85,0.000748
Yen->Yen,113081,67,0.000592


In [19]:
splits = build_walkforward_splits(v2a_df)

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )

fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


# Experiment with feature 1

In [20]:
V2A1_NUMERIC_FEATURES = BASE_NUMERIC_FEATURES.copy()

V2A1_CATEGORICAL_FEATURES = BASE_CATEGORICAL_FEATURES + ["currency_pair"]

FEATURE_COLUMNS_V2A1 = V2A1_NUMERIC_FEATURES + V2A1_CATEGORICAL_FEATURES

print("V1 feature count:", len(BASE_NUMERIC_FEATURES + BASE_CATEGORICAL_FEATURES))

print("V2-A1 feature count:", len(FEATURE_COLUMNS_V2A1))

print("\nAdded feature:")
print(
    set(FEATURE_COLUMNS_V2A1) - set(BASE_NUMERIC_FEATURES + BASE_CATEGORICAL_FEATURES)
)

V1 feature count: 12
V2-A1 feature count: 13

Added feature:
{'currency_pair'}


## Build preprocessing for V2-A1

In [21]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.utils.class_weight import compute_sample_weight

from lightgbm import LGBMClassifier

In [22]:
def make_v2a1_preprocessor():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                V2A1_NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                V2A1_CATEGORICAL_FEATURES,
            ),
        ],
        sparse_threshold=1.0,
    )

In [23]:
def make_v2a1_lightgbm_pipeline():

    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_v2a1_preprocessor(),
            ),
            (
                "model",
                model,
            ),
        ]
    )

In [24]:
TARGET_COL = "is_laundering"

In [25]:
splits = build_walkforward_splits(v2a_df)

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    y_train = v2a_df.iloc[train_idx][TARGET_COL]

    y_val = v2a_df.iloc[val_idx][TARGET_COL]

    print(
        fold.name,
        "| train:",
        f"{len(train_idx):,}",
        f"fraud={y_train.sum():,}",
        "| val:",
        f"{len(val_idx):,}",
        f"fraud={y_val.sum():,}",
    )

fold_1 | train: 2,076,752 fraud=1,121 | val: 207,430 fraud=407
fold_2 | train: 2,284,182 fraud=1,528 | val: 482,650 fraud=471
fold_3 | train: 2,766,832 fraud=1,999 | val: 964,840 fraud=1,028


## Run V2-A1 over all three folds

In [26]:
from time import perf_counter
import gc

v2a1_fold_results = []
v2a1_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    # -----------------------------------------
    # Fold data
    # -----------------------------------------

    X_train = v2a_df.iloc[train_idx][FEATURE_COLUMNS_V2A1]

    y_train = v2a_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v2a_df.iloc[val_idx][FEATURE_COLUMNS_V2A1]

    y_val = v2a_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    # -----------------------------------------
    # Same balanced weighting
    # -----------------------------------------

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    # -----------------------------------------
    # Fresh pipeline for each fold
    # -----------------------------------------

    pipeline = make_v2a1_lightgbm_pipeline()

    # -----------------------------------------
    # Fit
    # -----------------------------------------

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    # -----------------------------------------
    # Predict
    # -----------------------------------------

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    # -----------------------------------------
    # Shared evaluator
    # -----------------------------------------

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v2a1_fold_results.append(metrics)

    v2a1_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")

    print(f"PR lift   : {metrics['pr_lift']:.2f}x")

    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")

    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")

    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")

    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")

    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")

    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")

    print(f"Fit time  : {fit_seconds:.2f}s")

    # -----------------------------------------
    # Free memory
    # -----------------------------------------

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1
Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud



AP        : 0.062072
PR lift   : 31.64x
ROC-AUC   : 0.942580
Recall    : 80.0983%
Precision : 2.3219%
FP        : 13,714
Alerts    : 14,040
Alert rate: 6.7685%
Fit time  : 22.61s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.033860
PR lift   : 34.70x
ROC-AUC   : 0.890255
Recall    : 80.0425%
Precision : 0.5878%
FP        : 63,763
Alerts    : 64,140
Alert rate: 13.2891%
Fit time  : 23.00s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.044732
PR lift   : 41.98x
ROC-AUC   : 0.915019
Recall    : 80.0584%
Precision : 1.5530%
FP        : 52,170
Alerts    : 52,993
Alert rate: 5.4924%
Fit time  : 27.43s


In [27]:
v2a1_results = build_fold_metrics_table(v2a1_fold_results)

v2a1_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.062072,31.635192,0.942580,0.023219,0.800983,13714,14040,0.067685
1,fold_2,0.033860,34.697027,0.890255,0.005878,0.800425,63763,64140,0.132891
2,fold_3,0.044732,41.984083,0.915019,0.015530,0.800584,52170,52993,0.054924


In [28]:
# V1 - lightgbm result

In [29]:
v1_results = pd.DataFrame(
    {
        "fold": [
            "fold_1",
            "fold_2",
            "fold_3",
        ],
        "average_precision": [
            0.06307894817449604,
            0.032902,
            0.047368,
        ],
        "pr_lift": [
            32.148565650702,
            33.716026,
            44.457524,
        ],
        "roc_auc": [
            0.9419330682209686,
            0.887220,
            0.917476,
        ],
        "precision_at_recall_floor": [
            0.022770,
            0.005674,
            0.015732,
        ],
        "false_positives_at_recall_floor": [
            13991,
            66063,
            51490,
        ],
        "alert_rate_at_recall_floor": [
            0.069021,
            0.137657,
            0.054219,
        ],
    }
)

## Compare V1 vs V2-A1 directly

In [30]:
comparison = v1_results.merge(
    v2a1_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v1",
        "_v2a1",
    ),
)

In [31]:
comparison["ap_change"] = (
    comparison["average_precision_v2a1"] - comparison["average_precision_v1"]
)

comparison["ap_change_pct"] = (
    comparison["ap_change"] / comparison["average_precision_v1"] * 100
)

comparison["fp_change"] = (
    comparison["false_positives_at_recall_floor_v2a1"]
    - comparison["false_positives_at_recall_floor_v1"]
)

comparison["alert_rate_change"] = (
    comparison["alert_rate_at_recall_floor_v2a1"]
    - comparison["alert_rate_at_recall_floor_v1"]
)

In [32]:
comparison[
    [
        "fold",
        "average_precision_v1",
        "average_precision_v2a1",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v1",
        "precision_at_recall_floor_v2a1",
        "false_positives_at_recall_floor_v1",
        "false_positives_at_recall_floor_v2a1",
        "fp_change",
        "alert_rate_at_recall_floor_v1",
        "alert_rate_at_recall_floor_v2a1",
        "alert_rate_change",
    ]
]

,fold,average_precision_v1,average_precision_v2a1,ap_change,ap_change_pct,precision_at_recall_floor_v1,precision_at_recall_floor_v2a1,false_positives_at_recall_floor_v1,false_positives_at_recall_floor_v2a1,fp_change,alert_rate_at_recall_floor_v1,alert_rate_at_recall_floor_v2a1,alert_rate_change
0,fold_1,0.063079,0.062072,-0.001007,-1.596878,0.022770,0.023219,13991,13714,-277,0.069021,0.067685,-0.001336
1,fold_2,0.032902,0.033860,0.000958,2.910233,0.005674,0.005878,66063,63763,-2300,0.137657,0.132891,-0.004766
2,fold_3,0.047368,0.044732,-0.002636,-5.564031,0.015732,0.015530,51490,52170,680,0.054219,0.054924,0.000705


## V2-A1 — Currency Pair

**Decision: DROP**

Adding `currency_pair` produced inconsistent temporal results.

- Fold 1 AP decreased by 1.60%.
- Fold 2 AP improved by 2.91% and reduced false positives by 2,300.
- Fold 3 AP decreased by 5.56% and false positives increased by 680.
- Mean AP decreased slightly relative to V1.

Although the feature improved the difficult Fold 2, it did not provide stable
improvement across time and degraded both ranking quality and alert burden on
the latest Fold 3.

The existing `payment_currency` and `receiving_currency` features may already
allow LightGBM to learn most useful currency interactions directly.

`currency_pair` will therefore not be retained in the current feature set.

# Experiment 2- F2

In [33]:
def add_payment_format_currency_pair(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add a current-transaction categorical interaction:

        payment format
        + payment currency
        + receiving currency

    No historical or target information is used.
    """

    result = df.copy()

    result["payment_format_currency_pair"] = (
        result["payment_format"].astype(str)
        + "|"
        + result["payment_currency"].astype(str)
        + "->"
        + result["receiving_currency"].astype(str)
    )

    return result

In [34]:
v2a2_df = add_payment_format_currency_pair(dev_df)

In [35]:
v2a2_df[
    [
        "payment_format",
        "payment_currency",
        "receiving_currency",
        "payment_format_currency_pair",
    ]
].head(15)

,payment_format,payment_currency,receiving_currency,payment_format_currency_pair
0,Reinvestment,Shekel,Shekel,Reinvestment|Shekel->Shekel
1,Reinvestment,US Dollar,US Dollar,Reinvestment|US Dollar->US Dollar
2,Cheque,Euro,Euro,Cheque|Euro->Euro
3,Reinvestment,Swiss Franc,Swiss Franc,Reinvestment|Swiss Franc->Swiss Franc
4,Reinvestment,US Dollar,US Dollar,Reinvestment|US Dollar->US Dollar
5,Reinvestment,US Dollar,US Dollar,Reinvestment|US Dollar->US Dollar
6,Reinvestment,US Dollar,US Dollar,Reinvestment|US Dollar->US Dollar
7,Credit Card,US Dollar,US Dollar,Credit Card|US Dollar->US Dollar
8,Credit Card,Shekel,Shekel,Credit Card|Shekel->Shekel
9,Reinvestment,Euro,Euro,Reinvestment|Euro->Euro


In [36]:
print(
    "Unique payment-format/currency combinations:",
    v2a2_df["payment_format_currency_pair"].nunique(),
)

Unique payment-format/currency combinations: 303


In [37]:
V2A2_NUMERIC_FEATURES = BASE_NUMERIC_FEATURES.copy()

V2A2_CATEGORICAL_FEATURES = BASE_CATEGORICAL_FEATURES + ["payment_format_currency_pair"]

FEATURE_COLUMNS_V2A2 = V2A2_NUMERIC_FEATURES + V2A2_CATEGORICAL_FEATURES

In [38]:
V1_FEATURES = BASE_NUMERIC_FEATURES + BASE_CATEGORICAL_FEATURES

print("V1 feature count:", len(V1_FEATURES))

print("V2-A2 feature count:", len(FEATURE_COLUMNS_V2A2))

print("Added:", set(FEATURE_COLUMNS_V2A2) - set(V1_FEATURES))

V1 feature count: 12
V2-A2 feature count: 13
Added: {'payment_format_currency_pair'}


In [39]:
def make_v2a2_preprocessor():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                V2A2_NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                V2A2_CATEGORICAL_FEATURES,
            ),
        ],
        sparse_threshold=1.0,
    )

In [40]:
def make_v2a2_lightgbm_pipeline():

    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_v2a2_preprocessor(),
            ),
            (
                "model",
                model,
            ),
        ]
    )

In [41]:
splits = build_walkforward_splits(v2a2_df)

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )

fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [42]:
v2a2_fold_results = []
v2a2_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v2a2_df.iloc[train_idx][FEATURE_COLUMNS_V2A2]

    y_train = v2a2_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v2a2_df.iloc[val_idx][FEATURE_COLUMNS_V2A2]

    y_val = v2a2_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_v2a2_lightgbm_pipeline()

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v2a2_fold_results.append(metrics)

    v2a2_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")

    print(f"PR lift   : {metrics['pr_lift']:.2f}x")

    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")

    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")

    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")

    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")

    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")

    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")

    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1
Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud



AP        : 0.073319
PR lift   : 37.37x
ROC-AUC   : 0.945275
Recall    : 80.0983%
Precision : 2.3134%
FP        : 13,766
Alerts    : 14,092
Alert rate: 6.7936%
Fit time  : 26.34s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.033589
PR lift   : 34.42x
ROC-AUC   : 0.883714
Recall    : 80.0425%
Precision : 0.5363%
FP        : 69,926
Alerts    : 70,303
Alert rate: 14.5660%
Fit time  : 24.27s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.056538
PR lift   : 53.06x
ROC-AUC   : 0.913048
Recall    : 80.0584%
Precision : 1.5690%
FP        : 51,630
Alerts    : 52,453
Alert rate: 5.4364%
Fit time  : 32.52s


In [43]:
v2a2_results = build_fold_metrics_table(v2a2_fold_results)

v2a2_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.073319,37.367306,0.945275,0.023134,0.800983,13766,14092,0.067936
1,fold_2,0.033589,34.419981,0.883714,0.005363,0.800425,69926,70303,0.145660
2,fold_3,0.056538,53.064121,0.913048,0.015690,0.800584,51630,52453,0.054364


In [44]:
comparison_v2a2 = v1_results.merge(
    v2a2_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v1",
        "_v2a2",
    ),
)

In [45]:
comparison_v2a2["ap_change"] = (
    comparison_v2a2["average_precision_v2a2"] - comparison_v2a2["average_precision_v1"]
)

comparison_v2a2["ap_change_pct"] = (
    comparison_v2a2["ap_change"] / comparison_v2a2["average_precision_v1"] * 100
)

comparison_v2a2["fp_change"] = (
    comparison_v2a2["false_positives_at_recall_floor_v2a2"]
    - comparison_v2a2["false_positives_at_recall_floor_v1"]
)

comparison_v2a2["alert_rate_change"] = (
    comparison_v2a2["alert_rate_at_recall_floor_v2a2"]
    - comparison_v2a2["alert_rate_at_recall_floor_v1"]
)

In [46]:
comparison_v2a2[
    [
        "fold",
        "average_precision_v1",
        "average_precision_v2a2",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v1",
        "precision_at_recall_floor_v2a2",
        "false_positives_at_recall_floor_v1",
        "false_positives_at_recall_floor_v2a2",
        "fp_change",
        "alert_rate_at_recall_floor_v1",
        "alert_rate_at_recall_floor_v2a2",
        "alert_rate_change",
    ]
]

,fold,average_precision_v1,average_precision_v2a2,ap_change,ap_change_pct,precision_at_recall_floor_v1,precision_at_recall_floor_v2a2,false_positives_at_recall_floor_v1,false_positives_at_recall_floor_v2a2,fp_change,alert_rate_at_recall_floor_v1,alert_rate_at_recall_floor_v2a2,alert_rate_change
0,fold_1,0.063079,0.073319,0.010240,16.233197,0.022770,0.023134,13991,13766,-225,0.069021,0.067936,-0.001085
1,fold_2,0.032902,0.033589,0.000687,2.088523,0.005674,0.005363,66063,69926,3863,0.137657,0.145660,0.008003
2,fold_3,0.047368,0.056538,0.009170,19.358606,0.015732,0.015690,51490,51630,140,0.054219,0.054364,0.000145


### V2-A2 — Payment Format × Currency Pair

**Decision: KEEP PROVISIONALLY**

Adding `payment_format_currency_pair` improved Average Precision on all three
chronological validation folds:

- Fold 1: +16.23%
- Fold 2: +2.09%
- Fold 3: +19.36%

Mean Average Precision increased from approximately `0.047783` to `0.054482`,
an improvement of roughly 14%.

The feature therefore provides consistent additional ranking signal beyond the
existing payment-format and currency columns.

However, operational behavior at approximately 80% recall was mixed.

- Fold 1 false positives decreased by 225.
- Fold 2 false positives increased by 3,863.
- Fold 3 false positives increased by only 140.

Therefore, the feature should not be interpreted as a false-positive reduction
feature. It improves overall fraud ranking, but its effect on the high-recall
operating region is not uniformly positive.

Because Average Precision is the primary development metric and the ranking
improvement is consistent across all folds—especially the latest Fold 3—the
feature will be retained provisionally and reevaluated after additional feature
engineering and model tuning.

# Experiment - F3


In [47]:
def add_payment_format_bank_relation(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add a leakage-safe categorical interaction between
    payment format and same-bank/cross-bank status.
    """

    result = df.copy()

    bank_relation = np.where(
        result["same_bank_flag"] == 1,
        "same_bank",
        "cross_bank",
    )

    result["payment_format_bank_relation"] = (
        result["payment_format"].astype(str) + "|" + bank_relation
    )

    return result

In [48]:
v2a3_df = add_payment_format_bank_relation(dev_df)

In [49]:
v2a3_df[
    [
        "payment_format",
        "same_bank_flag",
        "payment_format_bank_relation",
    ]
].head(20)

,payment_format,same_bank_flag,payment_format_bank_relation
0,Reinvestment,1,Reinvestment|same_bank
1,Reinvestment,1,Reinvestment|same_bank
2,Cheque,0,Cheque|cross_bank
3,Reinvestment,1,Reinvestment|same_bank
4,Reinvestment,1,Reinvestment|same_bank
5,Reinvestment,1,Reinvestment|same_bank
6,Reinvestment,1,Reinvestment|same_bank
7,Credit Card,0,Credit Card|cross_bank
8,Credit Card,0,Credit Card|cross_bank
9,Reinvestment,1,Reinvestment|same_bank


In [50]:
print("Unique categories:", v2a3_df["payment_format_bank_relation"].nunique())

Unique categories: 13


In [51]:
V2A3_NUMERIC_FEATURES = BASE_NUMERIC_FEATURES.copy()

V2A3_CATEGORICAL_FEATURES = BASE_CATEGORICAL_FEATURES + ["payment_format_bank_relation"]

FEATURE_COLUMNS_V2A3 = V2A3_NUMERIC_FEATURES + V2A3_CATEGORICAL_FEATURES

In [52]:
V1_FEATURES = BASE_NUMERIC_FEATURES + BASE_CATEGORICAL_FEATURES

print("V1 features:", len(V1_FEATURES))
print("V2-A3 features:", len(FEATURE_COLUMNS_V2A3))

print("Added:", set(FEATURE_COLUMNS_V2A3) - set(V1_FEATURES))

V1 features: 12
V2-A3 features: 13
Added: {'payment_format_bank_relation'}


## Preprocessor

In [53]:
def make_v2a3_preprocessor():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                V2A3_NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                V2A3_CATEGORICAL_FEATURES,
            ),
        ],
        sparse_threshold=1.0,
    )

In [54]:
def make_v2a3_lightgbm_pipeline():

    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_v2a3_preprocessor(),
            ),
            (
                "model",
                model,
            ),
        ]
    )

In [55]:
splits = build_walkforward_splits(v2a3_df)

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )

fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [56]:
v2a3_fold_results = []
v2a3_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v2a3_df.iloc[train_idx][FEATURE_COLUMNS_V2A3]

    y_train = v2a3_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v2a3_df.iloc[val_idx][FEATURE_COLUMNS_V2A3]

    y_val = v2a3_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_v2a3_lightgbm_pipeline()

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v2a3_fold_results.append(metrics)

    v2a3_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")

    print(f"PR lift   : {metrics['pr_lift']:.2f}x")

    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")

    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")

    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")

    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")

    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")

    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")

    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1
Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud



AP        : 0.065998
PR lift   : 33.64x
ROC-AUC   : 0.943319
Recall    : 80.0983%
Precision : 2.2756%
FP        : 14,000
Alerts    : 14,326
Alert rate: 6.9064%
Fit time  : 22.76s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.030510
PR lift   : 31.26x
ROC-AUC   : 0.885097
Recall    : 80.0425%
Precision : 0.6299%
FP        : 59,471
Alerts    : 59,848
Alert rate: 12.3999%
Fit time  : 20.37s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.051462
PR lift   : 48.30x
ROC-AUC   : 0.915439
Recall    : 80.0584%
Precision : 1.5816%
FP        : 51,213
Alerts    : 52,036
Alert rate: 5.3932%
Fit time  : 24.97s


In [57]:
v2a3_results = build_fold_metrics_table(v2a3_fold_results)

v2a3_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.065998,33.636266,0.943319,0.022756,0.800983,14000,14326,0.069064
1,fold_2,0.030510,31.264281,0.885097,0.006299,0.800425,59471,59848,0.123999
2,fold_3,0.051462,48.300050,0.915439,0.015816,0.800584,51213,52036,0.053932


In [58]:
comparison_v2a3 = v1_results.merge(
    v2a3_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v1",
        "_v2a3",
    ),
)

In [59]:
comparison_v2a3["ap_change"] = (
    comparison_v2a3["average_precision_v2a3"] - comparison_v2a3["average_precision_v1"]
)

comparison_v2a3["ap_change_pct"] = (
    comparison_v2a3["ap_change"] / comparison_v2a3["average_precision_v1"] * 100
)

comparison_v2a3["fp_change"] = (
    comparison_v2a3["false_positives_at_recall_floor_v2a3"]
    - comparison_v2a3["false_positives_at_recall_floor_v1"]
)

comparison_v2a3["alert_rate_change"] = (
    comparison_v2a3["alert_rate_at_recall_floor_v2a3"]
    - comparison_v2a3["alert_rate_at_recall_floor_v1"]
)

In [60]:
comparison_v2a3[
    [
        "fold",
        "average_precision_v1",
        "average_precision_v2a3",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v1",
        "precision_at_recall_floor_v2a3",
        "false_positives_at_recall_floor_v1",
        "false_positives_at_recall_floor_v2a3",
        "fp_change",
        "alert_rate_at_recall_floor_v1",
        "alert_rate_at_recall_floor_v2a3",
        "alert_rate_change",
    ]
]

,fold,average_precision_v1,average_precision_v2a3,ap_change,ap_change_pct,precision_at_recall_floor_v1,precision_at_recall_floor_v2a3,false_positives_at_recall_floor_v1,false_positives_at_recall_floor_v2a3,fp_change,alert_rate_at_recall_floor_v1,alert_rate_at_recall_floor_v2a3,alert_rate_change
0,fold_1,0.063079,0.065998,0.002919,4.627581,0.022770,0.022756,13991,14000,9,0.069021,0.069064,0.000043
1,fold_2,0.032902,0.030510,-0.002392,-7.271179,0.005674,0.006299,66063,59471,-6592,0.137657,0.123999,-0.013658
2,fold_3,0.047368,0.051462,0.004094,8.642649,0.015732,0.015816,51490,51213,-277,0.054219,0.053932,-0.000287


### V2-A3 — Payment Format × Bank Relation

**Decision: KEEP PROVISIONALLY**

Adding `payment_format_bank_relation` produced mixed but useful results across
the chronological folds.

- Fold 1 Average Precision improved by 4.63%, with essentially unchanged
  false-positive burden.
- Fold 2 Average Precision decreased by 7.27%, but false positives at
  approximately 80% recall decreased by 6,592 and alert rate fell from
  13.77% to 12.40%.
- Fold 3 Average Precision improved by 8.64%, while false positives decreased
  by 277.

Mean Average Precision increased from approximately `0.047783` to `0.049323`.

The feature therefore appears to provide useful signal, particularly in the
high-recall operating region. Its Fold 2 behavior also shows why Average
Precision and operational alert metrics must both be evaluated: ranking quality
across the full PR curve declined, while false-positive burden at the target
recall level improved substantially.

Because the latest Fold 3 improved on both Average Precision and false-positive
burden, and the overall mean AP also increased, the feature will be retained
provisionally.

Its final value will be reassessed when combined with the other surviving
interaction feature.

# Experiment 4 - remove amount_paid and amount_received

In [61]:
V2B1_NUMERIC_FEATURES = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
    "log_amount_received",
    "log_amount_paid",
]

V2B1_CATEGORICAL_FEATURES = BASE_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V2B1 = V2B1_NUMERIC_FEATURES + V2B1_CATEGORICAL_FEATURES

In [62]:
V1_FEATURES = BASE_NUMERIC_FEATURES + BASE_CATEGORICAL_FEATURES

print("V1 feature count:", len(V1_FEATURES))
print("V2-B1 feature count:", len(FEATURE_COLUMNS_V2B1))

print("Removed:", set(V1_FEATURES) - set(FEATURE_COLUMNS_V2B1))

print("Added:", set(FEATURE_COLUMNS_V2B1) - set(V1_FEATURES))

V1 feature count: 12
V2-B1 feature count: 10
Removed: {'amount_received', 'amount_paid'}
Added: set()


In [63]:
def make_v2b1_preprocessor():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                V2B1_NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                V2B1_CATEGORICAL_FEATURES,
            ),
        ],
        sparse_threshold=1.0,
    )

In [64]:
def make_v2b1_lightgbm_pipeline():

    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_v2b1_preprocessor(),
            ),
            (
                "model",
                model,
            ),
        ]
    )

In [65]:
splits = build_walkforward_splits(dev_df)

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )

fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [66]:
v2b1_fold_results = []
v2b1_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = dev_df.iloc[train_idx][FEATURE_COLUMNS_V2B1]

    y_train = dev_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = dev_df.iloc[val_idx][FEATURE_COLUMNS_V2B1]

    y_val = dev_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_v2b1_lightgbm_pipeline()

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v2b1_fold_results.append(metrics)

    v2b1_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")

    print(f"PR lift   : {metrics['pr_lift']:.2f}x")

    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")

    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")

    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")

    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")

    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")

    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")

    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1


Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.068009
PR lift   : 34.66x
ROC-AUC   : 0.942114
Recall    : 80.0983%
Precision : 2.3965%
FP        : 13,277
Alerts    : 13,603
Alert rate: 6.5579%
Fit time  : 16.76s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.032947
PR lift   : 33.76x
ROC-AUC   : 0.889075
Recall    : 80.0425%
Precision : 0.7271%
FP        : 51,476
Alerts    : 51,853
Alert rate: 10.7434%
Fit time  : 19.92s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.045993
PR lift   : 43.17x
ROC-AUC   : 0.916700
Recall    : 80.0584%
Precision : 1.5568%
FP        : 52,041
Alerts    : 52,864
Alert rate: 5.4790%
Fit time  : 19.80s


In [67]:
v2b1_results = build_fold_metrics_table(v2b1_fold_results)

v2b1_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.068009,34.661288,0.942114,0.023965,0.800983,13277,13603,0.065579
1,fold_2,0.032947,33.761960,0.889075,0.007271,0.800425,51476,51853,0.107434
2,fold_3,0.045993,43.166891,0.916700,0.015568,0.800584,52041,52864,0.054790


In [68]:
comparison_v2b1 = v1_results.merge(
    v2b1_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v1",
        "_v2b1",
    ),
)

In [69]:
comparison_v2b1["ap_change"] = (
    comparison_v2b1["average_precision_v2b1"] - comparison_v2b1["average_precision_v1"]
)

comparison_v2b1["ap_change_pct"] = (
    comparison_v2b1["ap_change"] / comparison_v2b1["average_precision_v1"] * 100
)

comparison_v2b1["fp_change"] = (
    comparison_v2b1["false_positives_at_recall_floor_v2b1"]
    - comparison_v2b1["false_positives_at_recall_floor_v1"]
)

comparison_v2b1["alert_rate_change"] = (
    comparison_v2b1["alert_rate_at_recall_floor_v2b1"]
    - comparison_v2b1["alert_rate_at_recall_floor_v1"]
)

In [70]:
comparison_v2b1[
    [
        "fold",
        "average_precision_v1",
        "average_precision_v2b1",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v1",
        "precision_at_recall_floor_v2b1",
        "false_positives_at_recall_floor_v1",
        "false_positives_at_recall_floor_v2b1",
        "fp_change",
        "alert_rate_at_recall_floor_v1",
        "alert_rate_at_recall_floor_v2b1",
        "alert_rate_change",
    ]
]

,fold,average_precision_v1,average_precision_v2b1,ap_change,ap_change_pct,precision_at_recall_floor_v1,precision_at_recall_floor_v2b1,false_positives_at_recall_floor_v1,false_positives_at_recall_floor_v2b1,fp_change,alert_rate_at_recall_floor_v1,alert_rate_at_recall_floor_v2b1,alert_rate_change
0,fold_1,0.063079,0.068009,0.004930,7.815970,0.022770,0.023965,13991,13277,-714,0.069021,0.065579,-0.003442
1,fold_2,0.032902,0.032947,0.000045,0.136857,0.005674,0.007271,66063,51476,-14587,0.137657,0.107434,-0.030223
2,fold_3,0.047368,0.045993,-0.001375,-2.903508,0.015732,0.015568,51490,52041,551,0.054219,0.054790,0.000571


Removing raw amounts improves average temporal performance and substantially reduces false-positive burden in the difficult Fold 2, while causing a modest degradation on Fold 3.
### V2-B1 — Log-Only Amount Representation

**Decision: KEEP PROVISIONALLY AS NEW BASE**

Removing raw `amount_received` and `amount_paid` while retaining
`log_amount_received` and `log_amount_paid` produced the strongest overall
amount representation.

Mean Average Precision increased from approximately `0.047783` to `0.048983`.

At approximately 80% recall, total false positives across the three
chronological validation windows decreased from `131,544` to `116,794`,
a reduction of `14,750`.

The largest operational improvement occurred in Fold 2:

- AP remained effectively unchanged (`0.032902` → `0.032947`).
- False positives decreased from `66,063` to `51,476`.
- Alert rate decreased from `13.77%` to `10.74%`.

However, Fold 3 AP decreased by 2.90% and false positives increased by 551.
Therefore, log-only is not uniformly superior across time.

The log-only representation will be used provisionally as the new feature base,
while Fold 3 performance will continue to be monitored during subsequent
feature experiments.

# Experiment 5 - raw only

In [71]:
V2B2_NUMERIC_FEATURES = [
    "amount_received",
    "amount_paid",
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
]

V2B2_CATEGORICAL_FEATURES = BASE_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V2B2 = V2B2_NUMERIC_FEATURES + V2B2_CATEGORICAL_FEATURES

In [72]:
V1_FEATURES = BASE_NUMERIC_FEATURES + BASE_CATEGORICAL_FEATURES

print("V1 feature count:", len(V1_FEATURES))
print("V2-B2 feature count:", len(FEATURE_COLUMNS_V2B2))

print("Removed:", set(V1_FEATURES) - set(FEATURE_COLUMNS_V2B2))

print("Added:", set(FEATURE_COLUMNS_V2B2) - set(V1_FEATURES))

V1 feature count: 12
V2-B2 feature count: 10
Removed: {'log_amount_paid', 'log_amount_received'}
Added: set()


In [73]:
def make_v2b2_preprocessor():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                V2B2_NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                V2B2_CATEGORICAL_FEATURES,
            ),
        ],
        sparse_threshold=1.0,
    )

In [74]:
def make_v2b2_lightgbm_pipeline():

    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_v2b2_preprocessor(),
            ),
            (
                "model",
                model,
            ),
        ]
    )

In [75]:
v2b2_fold_results = []
v2b2_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = dev_df.iloc[train_idx][FEATURE_COLUMNS_V2B2]

    y_train = dev_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = dev_df.iloc[val_idx][FEATURE_COLUMNS_V2B2]

    y_val = dev_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_v2b2_lightgbm_pipeline()

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v2b2_fold_results.append(metrics)

    v2b2_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")

    print(f"PR lift   : {metrics['pr_lift']:.2f}x")

    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")

    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")

    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")

    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")

    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")

    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")

    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1
Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.060581
PR lift   : 30.88x
ROC-AUC   : 0.942707
Recall    : 80.0983%
Precision : 2.2692%
FP        : 14,040
Alerts    : 14,366
Alert rate: 6.9257%
Fit time  : 16.38s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.032990
PR lift   : 33.81x
ROC-AUC   : 0.889106
Recall    : 80.0425%
Precision : 0.6917%
FP        : 54,128
Alerts    : 54,505
Alert rate: 11.2929%
Fit time  : 16.34s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.045568
PR lift   : 42.77x
ROC-AUC   : 0.916493
Recall    : 80.0584%
Precision : 1.5382%
FP        : 52,682
Alerts    : 53,505
Alert rate: 5.5455%
Fit time  : 19.76s


In [76]:
v2b2_results = build_fold_metrics_table(v2b2_fold_results)

v2b2_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.060581,30.875399,0.942707,0.022692,0.800983,14040,14366,0.069257
1,fold_2,0.032990,33.805878,0.889106,0.006917,0.800425,54128,54505,0.112929
2,fold_3,0.045568,42.768087,0.916493,0.015382,0.800584,52682,53505,0.055455


In [77]:
comparison_v2b2 = v1_results.merge(
    v2b2_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v1",
        "_v2b2",
    ),
)

In [78]:
comparison_v2b2["ap_change"] = (
    comparison_v2b2["average_precision_v2b2"] - comparison_v2b2["average_precision_v1"]
)

comparison_v2b2["ap_change_pct"] = (
    comparison_v2b2["ap_change"] / comparison_v2b2["average_precision_v1"] * 100
)

comparison_v2b2["fp_change"] = (
    comparison_v2b2["false_positives_at_recall_floor_v2b2"]
    - comparison_v2b2["false_positives_at_recall_floor_v1"]
)

comparison_v2b2["alert_rate_change"] = (
    comparison_v2b2["alert_rate_at_recall_floor_v2b2"]
    - comparison_v2b2["alert_rate_at_recall_floor_v1"]
)

In [79]:
comparison_v2b2[
    [
        "fold",
        "average_precision_v1",
        "average_precision_v2b2",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v1",
        "precision_at_recall_floor_v2b2",
        "false_positives_at_recall_floor_v1",
        "false_positives_at_recall_floor_v2b2",
        "fp_change",
        "alert_rate_at_recall_floor_v1",
        "alert_rate_at_recall_floor_v2b2",
        "alert_rate_change",
    ]
]

,fold,average_precision_v1,average_precision_v2b2,ap_change,ap_change_pct,precision_at_recall_floor_v1,precision_at_recall_floor_v2b2,false_positives_at_recall_floor_v1,false_positives_at_recall_floor_v2b2,fp_change,alert_rate_at_recall_floor_v1,alert_rate_at_recall_floor_v2b2,alert_rate_change
0,fold_1,0.063079,0.060581,-0.002498,-3.960259,0.022770,0.022692,13991,14040,49,0.069021,0.069257,0.000236
1,fold_2,0.032902,0.032990,0.000088,0.267115,0.005674,0.006917,66063,54128,-11935,0.137657,0.112929,-0.024728
2,fold_3,0.047368,0.045568,-0.001800,-3.800549,0.015732,0.015382,51490,52682,1192,0.054219,0.055455,0.001236




### V2-B2 — Raw-Only Amount Representation

**Decision: DROP**

Removing `log_amount_received` and `log_amount_paid` while retaining only the
raw amount features did not improve LightGBM consistently.

- Fold 1 AP decreased by 3.96%.
- Fold 2 AP was effectively unchanged (+0.27%), although false positives
  decreased substantially.
- Fold 3 AP decreased by 3.80% and false positives increased by 1,192.
- Mean AP decreased from approximately `0.047783` to `0.046380`.

Raw-only therefore provides no advantage over either the original raw + log
representation or the log-only representation.

# Experiment 6- combined feeature set f2+f3

In [80]:
V2C1_NUMERIC_FEATURES = [
    "hour_of_day",
    "day_of_week",
    "is_weekend",
    "same_currency_flag",
    "same_bank_flag",
    "log_amount_received",
    "log_amount_paid",
]

V2C1_CATEGORICAL_FEATURES = [
    "receiving_currency",
    "payment_currency",
    "payment_format",
    "payment_format_currency_pair",
    "payment_format_bank_relation",
]

FEATURE_COLUMNS_V2C1 = V2C1_NUMERIC_FEATURES + V2C1_CATEGORICAL_FEATURES

In [81]:
print("V2-C1 feature count:", len(FEATURE_COLUMNS_V2C1))

print("\nNumeric:")
print(V2C1_NUMERIC_FEATURES)

print("\nCategorical:")
print(V2C1_CATEGORICAL_FEATURES)

V2-C1 feature count: 12

Numeric:
['hour_of_day', 'day_of_week', 'is_weekend', 'same_currency_flag', 'same_bank_flag', 'log_amount_received', 'log_amount_paid']

Categorical:
['receiving_currency', 'payment_currency', 'payment_format', 'payment_format_currency_pair', 'payment_format_bank_relation']


In [82]:
def add_v2c1_features(
    df: pd.DataFrame,
) -> pd.DataFrame:
    result = df.copy()

    result["payment_format_currency_pair"] = (
        result["payment_format"].astype(str)
        + "|"
        + result["payment_currency"].astype(str)
        + "->"
        + result["receiving_currency"].astype(str)
    )

    bank_relation = np.where(
        result["same_bank_flag"] == 1,
        "same_bank",
        "cross_bank",
    )

    result["payment_format_bank_relation"] = (
        result["payment_format"].astype(str) + "|" + bank_relation
    )

    return result

In [83]:
v2c1_df = add_v2c1_features(dev_df)

In [84]:
v2c1_df[
    [
        "payment_format",
        "payment_currency",
        "receiving_currency",
        "same_bank_flag",
        "payment_format_currency_pair",
        "payment_format_bank_relation",
    ]
].head()

,payment_format,payment_currency,receiving_currency,same_bank_flag,payment_format_currency_pair,payment_format_bank_relation
0,Reinvestment,Shekel,Shekel,1,Reinvestment|Shekel->Shekel,Reinvestment|same_bank
1,Reinvestment,US Dollar,US Dollar,1,Reinvestment|US Dollar->US Dollar,Reinvestment|same_bank
2,Cheque,Euro,Euro,0,Cheque|Euro->Euro,Cheque|cross_bank
3,Reinvestment,Swiss Franc,Swiss Franc,1,Reinvestment|Swiss Franc->Swiss Franc,Reinvestment|same_bank
4,Reinvestment,US Dollar,US Dollar,1,Reinvestment|US Dollar->US Dollar,Reinvestment|same_bank


In [85]:
def make_v2c1_preprocessor():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                V2C1_NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                V2C1_CATEGORICAL_FEATURES,
            ),
        ],
        sparse_threshold=1.0,
    )

In [86]:
def make_v2c1_lightgbm_pipeline():

    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_v2c1_preprocessor(),
            ),
            (
                "model",
                model,
            ),
        ]
    )


In [87]:
splits = build_walkforward_splits(v2c1_df)


In [88]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        len(train_idx),
        len(val_idx),
    )

fold_1 2076752 207430
fold_2 2284182 482650
fold_3 2766832 964840


In [89]:
v2c1_fold_results = []
v2c1_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v2c1_df.iloc[train_idx][FEATURE_COLUMNS_V2C1]

    y_train = v2c1_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v2c1_df.iloc[val_idx][FEATURE_COLUMNS_V2C1]

    y_val = v2c1_df.iloc[val_idx][TARGET_COL].to_numpy()

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_v2c1_lightgbm_pipeline()

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v2c1_fold_results.append(metrics)

    v2c1_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"AP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1


AP        : 0.067216
PR lift   : 34.26x
ROC-AUC   : 0.945093
Recall    : 80.0983%
Precision : 2.4736%
FP        : 12,853
Alerts    : 13,179
Alert rate: 6.3535%
Fit time  : 19.73s

Running fold_2
AP        : 0.033291
PR lift   : 34.11x
ROC-AUC   : 0.880317
Recall    : 80.0425%
Precision : 0.5165%
FP        : 72,619
Alerts    : 72,996
Alert rate: 15.1240%
Fit time  : 20.24s

Running fold_3
AP        : 0.055801
PR lift   : 52.37x
ROC-AUC   : 0.914341
Recall    : 80.0584%
Precision : 1.5356%
FP        : 52,770
Alerts    : 53,593
Alert rate: 5.5546%
Fit time  : 24.90s


In [90]:
v2c1_results = build_fold_metrics_table(v2c1_fold_results)

v2c1_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.067216,34.256825,0.945093,0.024736,0.800983,12853,13179,0.063535
1,fold_2,0.033291,34.114160,0.880317,0.005165,0.800425,72619,72996,0.151240
2,fold_3,0.055801,52.373009,0.914341,0.015356,0.800584,52770,53593,0.055546


## comparision with v1

In [91]:
comparison_v2c1 = v2b1_results.merge(
    v2c1_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_log_only",
        "_v2c1",
    ),
)

In [92]:
comparison_v2c1["ap_change"] = (
    comparison_v2c1["average_precision_v2c1"]
    - comparison_v2c1["average_precision_log_only"]
)

comparison_v2c1["ap_change_pct"] = (
    comparison_v2c1["ap_change"] / comparison_v2c1["average_precision_log_only"] * 100
)

comparison_v2c1["fp_change"] = (
    comparison_v2c1["false_positives_at_recall_floor_v2c1"]
    - comparison_v2c1["false_positives_at_recall_floor_log_only"]
)

comparison_v2c1["alert_rate_change"] = (
    comparison_v2c1["alert_rate_at_recall_floor_v2c1"]
    - comparison_v2c1["alert_rate_at_recall_floor_log_only"]
)

In [93]:
comparison_v2c1[
    [
        "fold",
        "average_precision_log_only",
        "average_precision_v2c1",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_log_only",
        "precision_at_recall_floor_v2c1",
        "false_positives_at_recall_floor_log_only",
        "false_positives_at_recall_floor_v2c1",
        "fp_change",
        "alert_rate_at_recall_floor_log_only",
        "alert_rate_at_recall_floor_v2c1",
        "alert_rate_change",
    ]
]

,fold,average_precision_log_only,average_precision_v2c1,ap_change,ap_change_pct,precision_at_recall_floor_log_only,precision_at_recall_floor_v2c1,false_positives_at_recall_floor_log_only,false_positives_at_recall_floor_v2c1,fp_change,alert_rate_at_recall_floor_log_only,alert_rate_at_recall_floor_v2c1,alert_rate_change
0,fold_1,0.068009,0.067216,-0.000794,-1.166901,0.023965,0.024736,13277,12853,-424,0.065579,0.063535,-0.002044
1,fold_2,0.032947,0.033291,0.000344,1.043186,0.007271,0.005165,51476,72619,21143,0.107434,0.151240,0.043806
2,fold_3,0.045993,0.055801,0.009809,21.326803,0.015568,0.015356,52041,52770,729,0.054790,0.055546,0.000756


### V2-C1 — Log-Only Base + Both Interaction Features

**Decision: DO NOT ADOPT AS FINAL FEATURE SET YET**

Combining `payment_format_currency_pair` and
`payment_format_bank_relation` on top of the log-only amount representation
improved overall ranking performance but produced unstable operational behavior.

Average Precision changes relative to the log-only base were:

- Fold 1: -1.17%
- Fold 2: +1.04%
- Fold 3: +21.33%

Mean Average Precision increased from approximately `0.048983` to `0.052103`.

However, Fold 2 false positives at approximately 80% recall increased from
`51,476` to `72,619`, an increase of `21,143`, while alert rate increased from
`10.74%` to `15.12%`.

The large Fold 3 AP improvement indicates that the interaction features contain
valuable ranking information, but their combined use creates unacceptable
temporal instability in the high-recall alert region.

The two interaction features will therefore be tested individually on top of
the log-only base before deciding which should be retained.

# Experiment 7 - Log-only + Payment Format × Currency Pair

In [94]:
v2c2_df = add_payment_format_currency_pair(dev_df)

In [95]:
V2C2_NUMERIC_FEATURES = V2B1_NUMERIC_FEATURES.copy()

V2C2_CATEGORICAL_FEATURES = V2B1_CATEGORICAL_FEATURES + ["payment_format_currency_pair"]

FEATURE_COLUMNS_V2C2 = V2C2_NUMERIC_FEATURES + V2C2_CATEGORICAL_FEATURES

In [96]:
print("V2-B1 feature count:", len(FEATURE_COLUMNS_V2B1))
print("V2-C2 feature count:", len(FEATURE_COLUMNS_V2C2))

print(
    "Added:",
    set(FEATURE_COLUMNS_V2C2) - set(FEATURE_COLUMNS_V2B1),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V2B1) - set(FEATURE_COLUMNS_V2C2),
)

V2-B1 feature count: 10
V2-C2 feature count: 11
Added: {'payment_format_currency_pair'}
Removed: set()


In [97]:
assert set(FEATURE_COLUMNS_V2C2) - set(FEATURE_COLUMNS_V2B1) == {
    "payment_format_currency_pair"
}

assert "amount_received" not in FEATURE_COLUMNS_V2C2
assert "amount_paid" not in FEATURE_COLUMNS_V2C2
assert "payment_format_bank_relation" not in FEATURE_COLUMNS_V2C2

In [98]:
print(
    "Unique payment-format/currency combinations:",
    v2c2_df["payment_format_currency_pair"].nunique(),
)

v2c2_df[
    [
        "payment_format",
        "payment_currency",
        "receiving_currency",
        "payment_format_currency_pair",
    ]
].head()

Unique payment-format/currency combinations: 303


,payment_format,payment_currency,receiving_currency,payment_format_currency_pair
0,Reinvestment,Shekel,Shekel,Reinvestment|Shekel->Shekel
1,Reinvestment,US Dollar,US Dollar,Reinvestment|US Dollar->US Dollar
2,Cheque,Euro,Euro,Cheque|Euro->Euro
3,Reinvestment,Swiss Franc,Swiss Franc,Reinvestment|Swiss Franc->Swiss Franc
4,Reinvestment,US Dollar,US Dollar,Reinvestment|US Dollar->US Dollar


## Preprocessor

In [99]:
def make_v2c2_preprocessor():

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                V2C2_NUMERIC_FEATURES,
            ),
            (
                "categorical",
                categorical_pipeline,
                V2C2_CATEGORICAL_FEATURES,
            ),
        ],
        sparse_threshold=1.0,
    )

In [100]:
def make_v2c2_lightgbm_pipeline():

    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                make_v2c2_preprocessor(),
            ),
            (
                "model",
                model,
            ),
        ]
    )

In [101]:
splits = build_walkforward_splits(v2c2_df)

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )

fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


## Run the three folds

In [102]:
v2c2_fold_results = []
v2c2_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v2c2_df.iloc[train_idx][FEATURE_COLUMNS_V2C2]
    y_train = v2c2_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v2c2_df.iloc[val_idx][FEATURE_COLUMNS_V2C2]
    y_val = v2c2_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")
    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_v2c2_lightgbm_pipeline()

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v2c2_fold_results.append(metrics)

    v2c2_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1


Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.073276
PR lift   : 37.35x
ROC-AUC   : 0.943206
Recall    : 80.0983%
Precision : 2.5531%
FP        : 12,443
Alerts    : 12,769
Alert rate: 6.1558%
Fit time  : 20.04s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.034781
PR lift   : 35.64x
ROC-AUC   : 0.887842
Recall    : 80.0425%
Precision : 0.6128%
FP        : 61,139
Alerts    : 61,516
Alert rate: 12.7455%
Fit time  : 18.54s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.055537
PR lift   : 52.13x
ROC-AUC   : 0.913820
Recall    : 80.0584%
Precision : 1.5380%
FP        : 52,687
Alerts    : 53,510
Alert rate: 5.5460%
Fit time  : 22.63s


In [103]:
v2c2_results = build_fold_metrics_table(v2c2_fold_results)

v2c2_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.073276,37.345706,0.943206,0.025531,0.800983,12443,12769,0.061558
1,fold_2,0.034781,35.641707,0.887842,0.006128,0.800425,61139,61516,0.127455
2,fold_3,0.055537,52.125103,0.913820,0.015380,0.800584,52687,53510,0.055460


In [104]:
comparison_v2c2 = v2b1_results.merge(
    v2c2_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_log_only",
        "_v2c2",
    ),
)

In [105]:
comparison_v2c2["ap_change"] = (
    comparison_v2c2["average_precision_v2c2"]
    - comparison_v2c2["average_precision_log_only"]
)

comparison_v2c2["ap_change_pct"] = (
    comparison_v2c2["ap_change"] / comparison_v2c2["average_precision_log_only"] * 100
)

comparison_v2c2["fp_change"] = (
    comparison_v2c2["false_positives_at_recall_floor_v2c2"]
    - comparison_v2c2["false_positives_at_recall_floor_log_only"]
)

comparison_v2c2["alert_rate_change"] = (
    comparison_v2c2["alert_rate_at_recall_floor_v2c2"]
    - comparison_v2c2["alert_rate_at_recall_floor_log_only"]
)

In [106]:
comparison_v2c2[
    [
        "fold",
        "average_precision_log_only",
        "average_precision_v2c2",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_log_only",
        "precision_at_recall_floor_v2c2",
        "false_positives_at_recall_floor_log_only",
        "false_positives_at_recall_floor_v2c2",
        "fp_change",
        "alert_rate_at_recall_floor_log_only",
        "alert_rate_at_recall_floor_v2c2",
        "alert_rate_change",
    ]
]

,fold,average_precision_log_only,average_precision_v2c2,ap_change,ap_change_pct,precision_at_recall_floor_log_only,precision_at_recall_floor_v2c2,false_positives_at_recall_floor_log_only,false_positives_at_recall_floor_v2c2,fp_change,alert_rate_at_recall_floor_log_only,alert_rate_at_recall_floor_v2c2,alert_rate_change
0,fold_1,0.068009,0.073276,0.005267,7.744715,0.023965,0.025531,13277,12443,-834,0.065579,0.061558,-0.004021
1,fold_2,0.032947,0.034781,0.001834,5.567646,0.007271,0.006128,51476,61139,9663,0.107434,0.127455,0.020021
2,fold_3,0.045993,0.055537,0.009545,20.752505,0.015568,0.015380,52041,52687,646,0.054790,0.055460,0.000670


### V2-C2 Decision — DROP as Candidate Base

**Feature tested:** Log-only base + `payment_format_currency_pair`

V2-C2 improved Average Precision across all three chronological folds:

* Fold 1: **+7.74% AP**
* Fold 2: **+5.57% AP**
* Fold 3: **+20.75% AP**

However, the improvement in ranking quality was not accompanied by stable operational improvement at the ≥80% recall operating point.

The main concern was **Fold 2**, where:

* False positives increased from **51,476 → 61,139** (**+9,663**)
* Precision decreased from **0.7271% → 0.6128%**
* Alert rate increased from **10.74% → 12.75%**

Fold 3 also showed a small increase in false positives (**+646**) despite the large AP improvement.

Therefore, although `payment_format_currency_pair` contains useful fraud-ranking signal, its benefit is **not temporally stable at the operational recall target**. In particular, the substantial increase in Fold 2 analyst burden conflicts with the project objective of improving fraud separation while controlling false positives.

**Decision:** Do not select V2-C2 as the transaction-level base. Retain the result as evidence that the interaction contains useful signal, but continue with the isolated `payment_format_bank_relation` ablation before selecting the final transaction-level feature set.


# Experiment 8 - Log-only + payment_format_bank_relation only

In [107]:
V2C3_NUMERIC_FEATURES = V2B1_NUMERIC_FEATURES.copy()

V2C3_CATEGORICAL_FEATURES = V2B1_CATEGORICAL_FEATURES + ["payment_format_bank_relation"]

FEATURE_COLUMNS_V2C3 = V2C3_NUMERIC_FEATURES + V2C3_CATEGORICAL_FEATURES

In [108]:
print("V2-B1 feature count:", len(FEATURE_COLUMNS_V2B1))
print("V2-C3 feature count:", len(FEATURE_COLUMNS_V2C3))

print(
    "Added:",
    set(FEATURE_COLUMNS_V2C3) - set(FEATURE_COLUMNS_V2B1),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V2B1) - set(FEATURE_COLUMNS_V2C3),
)

V2-B1 feature count: 10
V2-C3 feature count: 11
Added: {'payment_format_bank_relation'}
Removed: set()


In [109]:
print("Added:", set(FEATURE_COLUMNS_V2C3) - set(FEATURE_COLUMNS_V2B1))

print("Removed:", set(FEATURE_COLUMNS_V2B1) - set(FEATURE_COLUMNS_V2C3))

assert set(FEATURE_COLUMNS_V2C3) - set(FEATURE_COLUMNS_V2B1) == {
    "payment_format_bank_relation"
}

assert "payment_format_currency_pair" not in FEATURE_COLUMNS_V2C3
assert "amount_received" not in FEATURE_COLUMNS_V2C3
assert "amount_paid" not in FEATURE_COLUMNS_V2C3

Added: {'payment_format_bank_relation'}
Removed: set()


## Preprocessor

In [110]:
def make_preprocessor(num_features, cat_features):

    numeric_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="median"),
            ),
            (
                "scaler",
                StandardScaler(),
            ),
        ]
    )

    categorical_pipeline = Pipeline(
        steps=[
            (
                "imputer",
                SimpleImputer(strategy="most_frequent"),
            ),
            (
                "onehot",
                OneHotEncoder(
                    handle_unknown="ignore",
                    sparse_output=True,
                ),
            ),
        ]
    )

    return ColumnTransformer(
        transformers=[
            (
                "numeric",
                numeric_pipeline,
                num_features,
            ),
            (
                "categorical",
                categorical_pipeline,
                cat_features,
            ),
        ],
        sparse_threshold=1.0,
    )

In [111]:
def make_lightgbm_pipeline(preprocessor, num_features, cat_features):

    model = LGBMClassifier(
        objective="binary",
        n_estimators=300,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=100,
        reg_lambda=1.0,
        random_state=42,
        n_jobs=-1,
        verbosity=-1,
    )

    return Pipeline(
        steps=[
            (
                "preprocessor",
                preprocessor(num_features, cat_features),
            ),
            (
                "model",
                model,
            ),
        ]
    )

In [112]:
v2c3_df = add_payment_format_bank_relation(dev_df)

In [113]:
splits = build_walkforward_splits(v2c3_df)

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )

fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [114]:
v2c3_df[
    [
        "payment_format",
        "same_bank_flag",
        "payment_format_bank_relation",
    ]
].head(20)

,payment_format,same_bank_flag,payment_format_bank_relation
0,Reinvestment,1,Reinvestment|same_bank
1,Reinvestment,1,Reinvestment|same_bank
2,Cheque,0,Cheque|cross_bank
3,Reinvestment,1,Reinvestment|same_bank
4,Reinvestment,1,Reinvestment|same_bank
5,Reinvestment,1,Reinvestment|same_bank
6,Reinvestment,1,Reinvestment|same_bank
7,Credit Card,0,Credit Card|cross_bank
8,Credit Card,0,Credit Card|cross_bank
9,Reinvestment,1,Reinvestment|same_bank


In [115]:
print(
    "Unique categories:",
    v2c3_df["payment_format_bank_relation"].nunique(),
)

print(v2c3_df["payment_format_bank_relation"].value_counts())

Unique categories: 13
payment_format_bank_relation
Cheque|cross_bank         1285955
Credit Card|cross_bank     909693
Reinvestment|same_bank     481056
ACH|cross_bank             362132
Cash|cross_bank            339374
Wire|cross_bank            119079
Bitcoin|cross_bank          81276
ACH|same_bank               59923
Cheque|same_bank            32809
Bitcoin|same_bank           26139
Credit Card|same_bank       21977
Cash|same_bank               8102
Wire|same_bank               4157
Name: count, dtype: int64


## Run the folds

In [116]:
v2c3_fold_results = []
v2c3_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v2c3_df.iloc[train_idx][FEATURE_COLUMNS_V2C3]
    y_train = v2c3_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v2c3_df.iloc[val_idx][FEATURE_COLUMNS_V2C3]
    y_val = v2c3_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_lightgbm_pipeline(
        make_preprocessor, V2C3_NUMERIC_FEATURES, V2C3_CATEGORICAL_FEATURES
    )

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v2c3_fold_results.append(metrics)

    v2c3_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1
Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud



AP        : 0.063134
PR lift   : 32.18x
ROC-AUC   : 0.942259
Recall    : 80.0983%
Precision : 2.2784%
FP        : 13,982
Alerts    : 14,308
Alert rate: 6.8977%
Fit time  : 18.41s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.031627
PR lift   : 32.41x
ROC-AUC   : 0.886588
Recall    : 80.0425%
Precision : 0.5673%
FP        : 66,082
Alerts    : 66,459
Alert rate: 13.7696%
Fit time  : 22.19s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.047472
PR lift   : 44.56x
ROC-AUC   : 0.914613
Recall    : 80.0584%
Precision : 1.4931%
FP        : 54,299
Alerts    : 55,122
Alert rate: 5.7131%
Fit time  : 25.08s


In [117]:
v2c3_results = build_fold_metrics_table(v2c3_fold_results)

v2c3_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.063134,32.176826,0.942259,0.022784,0.800983,13982,14308,0.068977
1,fold_2,0.031627,32.409562,0.886588,0.005673,0.800425,66082,66459,0.137696
2,fold_3,0.047472,44.555487,0.914613,0.014931,0.800584,54299,55122,0.057131


In [118]:
comparison_v2c3 = v2b1_results.merge(
    v2c3_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_log_only",
        "_v2c3",
    ),
)

In [119]:
comparison_v2c3["ap_change"] = (
    comparison_v2c3["average_precision_v2c3"]
    - comparison_v2c3["average_precision_log_only"]
)

comparison_v2c3["ap_change_pct"] = (
    comparison_v2c3["ap_change"] / comparison_v2c3["average_precision_log_only"] * 100
)

comparison_v2c3["fp_change"] = (
    comparison_v2c3["false_positives_at_recall_floor_v2c3"]
    - comparison_v2c3["false_positives_at_recall_floor_log_only"]
)

comparison_v2c3["alert_rate_change"] = (
    comparison_v2c3["alert_rate_at_recall_floor_v2c3"]
    - comparison_v2c3["alert_rate_at_recall_floor_log_only"]
)

In [120]:
comparison_v2c3[
    [
        "fold",
        "average_precision_log_only",
        "average_precision_v2c3",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_log_only",
        "precision_at_recall_floor_v2c3",
        "false_positives_at_recall_floor_log_only",
        "false_positives_at_recall_floor_v2c3",
        "fp_change",
        "alert_rate_at_recall_floor_log_only",
        "alert_rate_at_recall_floor_v2c3",
        "alert_rate_change",
    ]
]

,fold,average_precision_log_only,average_precision_v2c3,ap_change,ap_change_pct,precision_at_recall_floor_log_only,precision_at_recall_floor_v2c3,false_positives_at_recall_floor_log_only,false_positives_at_recall_floor_v2c3,fp_change,alert_rate_at_recall_floor_log_only,alert_rate_at_recall_floor_v2c3,alert_rate_change
0,fold_1,0.068009,0.063134,-0.004875,-7.167830,0.023965,0.022784,13277,13982,705,0.065579,0.068977,0.003399
1,fold_2,0.032947,0.031627,-0.001320,-4.005688,0.007271,0.005673,51476,66082,14606,0.107434,0.137696,0.030262
2,fold_3,0.045993,0.047472,0.001479,3.216807,0.015568,0.014931,52041,54299,2258,0.054790,0.057131,0.002340


## V2-C3 Decision — DROP

**Feature tested:** Log-only base + `payment_format_bank_relation`

V2-C3 did not provide a stable improvement over the log-only V2-B1 base.

Average Precision decreased in the first two chronological folds:

* Fold 1: **−7.17%**
* Fold 2: **−4.01%**

Fold 3 showed only a small AP improvement of **+3.22%**.

Operational performance also deteriorated across all three folds. False positives increased by:

* Fold 1: **+705**
* Fold 2: **+14,606**
* Fold 3: **+2,258**

The largest deterioration occurred in Fold 2, where precision at the ≥80% recall operating point fell from **0.7271% to 0.5673%**, while alert rate increased from **10.74% to 13.77%**.

The small Fold 3 AP improvement is not sufficient to offset the ranking degradation in Folds 1–2 and the consistently higher false-positive burden.

**Decision:** Drop `payment_format_bank_relation` from the candidate transaction-level feature set.


# Experiment 9 - Sender velocity

In [121]:
def add_sender_tx_count_1h(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Count transactions made by the same sender during the
    previous 1 hour.

    Window:
        [t - 1 hour, t)

    Therefore:
    - current transaction is excluded
    - all transactions at the exact current timestamp are excluded
    - only strictly earlier activity is used

    Sender identity:
        from_bank_id + from_account_id
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    sender_cols = [
        "from_bank_id",
        "from_account_id",
    ]

    # ---------------------------------------------------------
    # 1. Count how many transactions each sender made at each
    #    exact timestamp.
    # ---------------------------------------------------------

    timestamp_counts = (
        result.groupby(
            sender_cols + ["transaction_timestamp"],
            dropna=False,
        )
        .size()
        .rename("tx_at_timestamp")
        .reset_index()
    )

    timestamp_counts = timestamp_counts.sort_values(
        sender_cols + ["transaction_timestamp"]
    )

    # ---------------------------------------------------------
    # 2. For every sender/timestamp, sum transactions from
    #    the previous hour, excluding this exact timestamp.
    # ---------------------------------------------------------

    rolling_counts = (
        timestamp_counts.set_index("transaction_timestamp")
        .groupby(
            sender_cols,
            dropna=False,
        )["tx_at_timestamp"]
        .rolling(
            window="1h",
            closed="left",
        )
        .sum()
        .fillna(0)
        .rename("sender_tx_count_1h")
        .reset_index()
    )

    rolling_counts["sender_tx_count_1h"] = rolling_counts["sender_tx_count_1h"].astype(
        "int32"
    )

    # ---------------------------------------------------------
    # 3. Merge the historical count back onto every transaction.
    #
    # All transactions from the same sender at the same exact
    # timestamp receive the same past-history count.
    # ---------------------------------------------------------

    result = result.merge(
        rolling_counts[
            sender_cols
            + [
                "transaction_timestamp",
                "sender_tx_count_1h",
            ]
        ],
        on=sender_cols + ["transaction_timestamp"],
        how="left",
        validate="many_to_one",
        sort=False,
    )

    return result

In [122]:
v3a1_df = add_sender_tx_count_1h(dev_df)

In [123]:
assert v3a1_df["transaction_timestamp"].max() < pd.Timestamp("2022-09-08")

In [124]:
v3a1_df[
    [
        "transaction_timestamp",
        "from_bank_id",
        "from_account_id",
        "sender_tx_count_1h",
    ]
].sort_values(by="sender_tx_count_1h", ascending=False).head(20)

,transaction_timestamp,from_bank_id,from_account_id,sender_tx_count_1h
344395,2022-09-01 01:00:00,70,100428660,2829
344411,2022-09-01 01:00:00,70,100428660,2829
344372,2022-09-01 01:00:00,70,100428660,2829
344501,2022-09-01 01:00:00,70,100428660,2829
344523,2022-09-01 01:00:00,70,100428660,2829
344554,2022-09-01 01:00:00,70,100428660,2829
344557,2022-09-01 01:00:00,70,100428660,2829
344290,2022-09-01 01:00:00,70,100428660,2829
344248,2022-09-01 01:00:00,70,100428660,2829
344232,2022-09-01 01:00:00,70,100428660,2829


In [125]:
v3a1_df["sender_tx_count_1h"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.731672e+06
mean     3.950252e+01
std      1.692505e+02
min      0.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      4.000000e+00
95%      2.030000e+02
99%      8.860000e+02
99.9%    1.182000e+03
max      2.829000e+03
Name: sender_tx_count_1h, dtype: float64

In [126]:
print(
    "Zero-history transactions:",
    (v3a1_df["sender_tx_count_1h"] == 0).sum(),
)

print(
    "Maximum 1h sender count:",
    v3a1_df["sender_tx_count_1h"].max(),
)

Zero-history transactions: 1864727
Maximum 1h sender count: 2829


In [127]:
sender_cols = [
    "from_bank_id",
    "from_account_id",
]

first_sender_timestamp = v3a1_df.groupby(sender_cols)[
    "transaction_timestamp"
].transform("min")

first_timestamp_mask = v3a1_df["transaction_timestamp"] == first_sender_timestamp

bad_first_rows = v3a1_df.loc[
    first_timestamp_mask & (v3a1_df["sender_tx_count_1h"] != 0),
    [
        "transaction_timestamp",
        "from_bank_id",
        "from_account_id",
        "sender_tx_count_1h",
    ],
]

print("Bad first rows:", len(bad_first_rows))
bad_first_rows.head(20)

Bad first rows: 0


,transaction_timestamp,from_bank_id,from_account_id,sender_tx_count_1h


In [128]:
assert v3a1_df["sender_tx_count_1h"].isna().sum() == 0

print("✅ No missing sender velocity values.")

✅ No missing sender velocity values.


In [129]:
assert (v3a1_df["sender_tx_count_1h"] >= 0).all()

print("✅ All velocity counts are non-negative.")

✅ All velocity counts are non-negative.


In [130]:
v3a1_df["sender_tx_count_1h"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.731672e+06
mean     3.950252e+01
std      1.692505e+02
min      0.000000e+00
50%      1.000000e+00
75%      1.000000e+00
90%      4.000000e+00
95%      2.030000e+02
99%      8.860000e+02
99.9%    1.182000e+03
max      2.829000e+03
Name: sender_tx_count_1h, dtype: float64

In [131]:
print(
    "Zero-history transactions:",
    (v3a1_df["sender_tx_count_1h"] == 0).sum(),
)

print(
    "Maximum 1h sender count:",
    v3a1_df["sender_tx_count_1h"].max(),
)

Zero-history transactions: 1864727
Maximum 1h sender count: 2829


In [132]:
active_sender = (
    v3a1_df.groupby(
        [
            "from_bank_id",
            "from_account_id",
        ]
    )
    .size()
    .sort_values(ascending=False)
    .index[0]
)

active_sender

('70', '100428660')

In [133]:
bank_id, account_id = active_sender

v3a1_df.loc[
    (v3a1_df["from_bank_id"] == bank_id) & (v3a1_df["from_account_id"] == account_id),
    [
        "transaction_timestamp",
        "sender_tx_count_1h",
        "is_laundering",
    ],
].sort_values("transaction_timestamp").head(30)

,transaction_timestamp,sender_tx_count_1h,is_laundering
7,2022-09-01,0,0
7470,2022-09-01,0,0
7413,2022-09-01,0,0
7336,2022-09-01,0,0
7151,2022-09-01,0,0
6999,2022-09-01,0,0
6399,2022-09-01,0,0
6361,2022-09-01,0,0
6359,2022-09-01,0,0
6325,2022-09-01,0,0


In [134]:
V3A1_NUMERIC_FEATURES = V2B1_NUMERIC_FEATURES + ["sender_tx_count_1h"]

V3A1_CATEGORICAL_FEATURES = V2B1_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V3A1 = V3A1_NUMERIC_FEATURES + V3A1_CATEGORICAL_FEATURES

In [135]:
print(
    "Added:",
    set(FEATURE_COLUMNS_V3A1) - set(FEATURE_COLUMNS_V2B1),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V2B1) - set(FEATURE_COLUMNS_V3A1),
)

Added: {'sender_tx_count_1h'}
Removed: set()


In [136]:
assert set(FEATURE_COLUMNS_V3A1) - set(FEATURE_COLUMNS_V2B1) == {"sender_tx_count_1h"}

assert set(FEATURE_COLUMNS_V2B1) - set(FEATURE_COLUMNS_V3A1) == set()

In [137]:
splits = build_walkforward_splits(v3a1_df)

## Preprocessor

In [138]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )

fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [139]:
v3a1_fold_results = []
v3a1_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v3a1_df.iloc[train_idx][FEATURE_COLUMNS_V3A1]
    y_train = v3a1_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v3a1_df.iloc[val_idx][FEATURE_COLUMNS_V3A1]
    y_val = v3a1_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_lightgbm_pipeline(
        make_preprocessor, V3A1_NUMERIC_FEATURES, V3A1_CATEGORICAL_FEATURES
    )

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v3a1_fold_results.append(metrics)

    v3a1_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1


Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.099760
PR lift   : 50.84x
ROC-AUC   : 0.964417
Recall    : 80.0983%
Precision : 2.8147%
FP        : 11,256
Alerts    : 11,582
Alert rate: 5.5836%
Fit time  : 17.76s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.060973
PR lift   : 62.48x
ROC-AUC   : 0.959666
Recall    : 80.0425%
Precision : 0.9307%
FP        : 40,128
Alerts    : 40,505
Alert rate: 8.3922%
Fit time  : 17.97s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.095932
PR lift   : 90.04x
ROC-AUC   : 0.969876
Recall    : 80.0584%
Precision : 1.9294%
FP        : 41,833
Alerts    : 42,656
Alert rate: 4.4210%
Fit time  : 26.63s


In [140]:
v3a1_results = build_fold_metrics_table(v3a1_fold_results)

v3a1_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.099760,50.843522,0.964417,0.028147,0.800983,11256,11582,0.055836
1,fold_2,0.060973,62.481408,0.959666,0.009307,0.800425,40128,40505,0.083922
2,fold_3,0.095932,90.038411,0.969876,0.019294,0.800584,41833,42656,0.044210


In [141]:
comparison_v3a1 = v2b1_results.merge(
    v3a1_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_log_only",
        "_v3a1",
    ),
)

In [142]:
comparison_v3a1["ap_change"] = (
    comparison_v3a1["average_precision_v3a1"]
    - comparison_v3a1["average_precision_log_only"]
)

comparison_v3a1["ap_change_pct"] = (
    comparison_v3a1["ap_change"] / comparison_v3a1["average_precision_log_only"] * 100
)

comparison_v3a1["fp_change"] = (
    comparison_v3a1["false_positives_at_recall_floor_v3a1"]
    - comparison_v3a1["false_positives_at_recall_floor_log_only"]
)

comparison_v3a1["alert_rate_change"] = (
    comparison_v3a1["alert_rate_at_recall_floor_v3a1"]
    - comparison_v3a1["alert_rate_at_recall_floor_log_only"]
)

In [143]:
comparison_v3a1[
    [
        "fold",
        "average_precision_log_only",
        "average_precision_v3a1",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_log_only",
        "precision_at_recall_floor_v3a1",
        "false_positives_at_recall_floor_log_only",
        "false_positives_at_recall_floor_v3a1",
        "fp_change",
        "alert_rate_at_recall_floor_log_only",
        "alert_rate_at_recall_floor_v3a1",
        "alert_rate_change",
    ]
]

,fold,average_precision_log_only,average_precision_v3a1,ap_change,ap_change_pct,precision_at_recall_floor_log_only,precision_at_recall_floor_v3a1,false_positives_at_recall_floor_log_only,false_positives_at_recall_floor_v3a1,fp_change,alert_rate_at_recall_floor_log_only,alert_rate_at_recall_floor_v3a1,alert_rate_change
0,fold_1,0.068009,0.099760,0.031751,46.686764,0.023965,0.028147,13277,11256,-2021,0.065579,0.055836,-0.009743
1,fold_2,0.032947,0.060973,0.028026,85.064515,0.007271,0.009307,51476,40128,-11348,0.107434,0.083922,-0.023512
2,fold_3,0.045993,0.095932,0.049940,108.582105,0.015568,0.019294,52041,41833,-10208,0.054790,0.044210,-0.010580


Amazingg!!!
### V3-A1 Decision — KEEP

**Feature tested:** Log-only base + `sender_tx_count_1h`

Adding sender transaction velocity over the previous hour produced a strong and temporally consistent improvement across all three chronological validation folds.

Average Precision improved substantially:

* Fold 1: **+46.69%**
* Fold 2: **+85.06%**
* Fold 3: **+108.58%**

The ranking improvement was also accompanied by better operational performance at the ≥80% recall operating point.

False positives decreased by:

* Fold 1: **−2,021**
* Fold 2: **−11,348**
* Fold 3: **−10,208**

Precision improved in all three folds, while alert rate decreased consistently.

This is notably different from the earlier categorical interaction experiments, where AP improvements were sometimes accompanied by increased false-positive burden. `sender_tx_count_1h` improves both overall fraud ranking and the quality of the alert population.

The result provides strong evidence that recent sender behavior contains information that is not captured by transaction-level attributes alone.

**Decision:** KEEP `sender_tx_count_1h` and promote V3-A1 as the new provisional feature base for subsequent behavioral-feature ablations.


In [144]:
assert (v3a1_df.loc[first_timestamp_mask, "sender_tx_count_1h"] == 0).all()

assert v3a1_df["sender_tx_count_1h"].notna().all()

assert (v3a1_df["sender_tx_count_1h"] >= 0).all()

# Experiment 10 -  velocity window (24 hr)

In [145]:
def add_sender_tx_count_24h(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Count transactions made by the same sender during the
    previous 24 hours.

    Window:
        [t - 24 hours, t)

    Current transaction and all transactions at the exact
    current timestamp are excluded.

    Sender identity:
        from_bank_id + from_account_id
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()
    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    sender_cols = [
        "from_bank_id",
        "from_account_id",
    ]

    # Number of transactions from each sender
    # at each exact timestamp.
    timestamp_counts = (
        result.groupby(
            sender_cols + ["transaction_timestamp"],
            dropna=False,
        )
        .size()
        .rename("tx_at_timestamp")
        .reset_index()
    )

    timestamp_counts = timestamp_counts.sort_values(
        sender_cols + ["transaction_timestamp"]
    )

    # Sum transactions strictly before the current timestamp
    # over the previous 24 hours.
    rolling_counts = (
        timestamp_counts.set_index("transaction_timestamp")
        .groupby(
            sender_cols,
            dropna=False,
        )["tx_at_timestamp"]
        .rolling(
            window="24h",
            closed="left",
        )
        .sum()
        .fillna(0)
        .rename("sender_tx_count_24h")
        .reset_index()
    )

    rolling_counts["sender_tx_count_24h"] = rolling_counts[
        "sender_tx_count_24h"
    ].astype("int32")

    result = result.merge(
        rolling_counts[
            sender_cols
            + [
                "transaction_timestamp",
                "sender_tx_count_24h",
            ]
        ],
        on=sender_cols + ["transaction_timestamp"],
        how="left",
        validate="many_to_one",
        sort=False,
    )

    result = (
        result.sort_values("_row_order")
        .drop(columns="_row_order")
        .reset_index(drop=True)
    )

    return result

In [146]:
v3a2_df = add_sender_tx_count_24h(v3a1_df)

In [147]:
assert v3a2_df["sender_tx_count_24h"].notna().all()

assert (v3a2_df["sender_tx_count_24h"] >= 0).all()

assert (v3a2_df["sender_tx_count_24h"] >= v3a2_df["sender_tx_count_1h"]).all()

print("✅ Basic 24h velocity checks passed.")

✅ Basic 24h velocity checks passed.


In [148]:
sender_cols = [
    "from_bank_id",
    "from_account_id",
]

first_sender_timestamp = v3a2_df.groupby(sender_cols)[
    "transaction_timestamp"
].transform("min")

first_timestamp_mask = v3a2_df["transaction_timestamp"] == first_sender_timestamp

assert (
    v3a2_df.loc[
        first_timestamp_mask,
        "sender_tx_count_24h",
    ]
    == 0
).all()

print("✅ First-timestamp 24h leakage check passed.")

✅ First-timestamp 24h leakage check passed.


In [149]:
v3a2_df["sender_tx_count_24h"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.731672e+06
mean     7.915975e+02
std      3.455600e+03
min      0.000000e+00
50%      3.000000e+00
75%      8.000000e+00
90%      1.900000e+01
95%      3.357000e+03
99%      1.876700e+04
99.9%    2.562500e+04
max      2.636800e+04
Name: sender_tx_count_24h, dtype: float64

In [150]:
v3a2_df[
    [
        "sender_tx_count_1h",
        "sender_tx_count_24h",
    ]
].describe(
    percentiles=[
        0.50,
        0.90,
        0.95,
        0.99,
    ]
)

,sender_tx_count_1h,sender_tx_count_24h
count,3.731672e+06,3.731672e+06
mean,3.950252e+01,7.915975e+02
std,1.692505e+02,3.455600e+03
min,0.000000e+00,0.000000e+00
50%,1.000000e+00,3.000000e+00
90%,4.000000e+00,1.900000e+01
95%,2.030000e+02,3.357000e+03
99%,8.860000e+02,1.876700e+04
max,2.829000e+03,2.636800e+04


## V3-A2

In [151]:
V3A2_NUMERIC_FEATURES = V3A1_NUMERIC_FEATURES + ["sender_tx_count_24h"]

V3A2_CATEGORICAL_FEATURES = V3A1_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V3A2 = V3A2_NUMERIC_FEATURES + V3A2_CATEGORICAL_FEATURES

In [152]:
print(
    "Added:",
    set(FEATURE_COLUMNS_V3A2) - set(FEATURE_COLUMNS_V3A1),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V3A1) - set(FEATURE_COLUMNS_V3A2),
)

Added: {'sender_tx_count_24h'}
Removed: set()


In [153]:
assert set(FEATURE_COLUMNS_V3A2) - set(FEATURE_COLUMNS_V3A1) == {"sender_tx_count_24h"}

assert set(FEATURE_COLUMNS_V3A1) - set(FEATURE_COLUMNS_V3A2) == set()

In [154]:
splits = build_walkforward_splits(v3a2_df)
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )


fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [155]:
v3a2_fold_results = []
v3a2_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v3a2_df.iloc[train_idx][FEATURE_COLUMNS_V3A2]
    y_train = v3a2_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v3a2_df.iloc[val_idx][FEATURE_COLUMNS_V3A2]
    y_val = v3a2_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_lightgbm_pipeline(
        make_preprocessor, V3A2_NUMERIC_FEATURES, V3A2_CATEGORICAL_FEATURES
    )

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v3a2_fold_results.append(metrics)

    v3a2_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1


Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.086501
PR lift   : 44.09x
ROC-AUC   : 0.965292
Recall    : 80.0983%
Precision : 2.9050%
FP        : 10,896
Alerts    : 11,222
Alert rate: 5.4100%
Fit time  : 20.22s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.053027
PR lift   : 54.34x
ROC-AUC   : 0.957117
Recall    : 80.0425%
Precision : 0.8552%
FP        : 43,705
Alerts    : 44,082
Alert rate: 9.1333%
Fit time  : 23.61s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.090941
PR lift   : 85.35x
ROC-AUC   : 0.969496
Recall    : 80.0584%
Precision : 1.8366%
FP        : 43,989
Alerts    : 44,812
Alert rate: 4.6445%
Fit time  : 28.74s


In [156]:
v3a2_results = build_fold_metrics_table(v3a2_fold_results)

v3a2_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.086501,44.085855,0.965292,0.029050,0.800983,10896,11222,0.054100
1,fold_2,0.053027,54.338200,0.957117,0.008552,0.800425,43705,44082,0.091333
2,fold_3,0.090941,85.353588,0.969496,0.018366,0.800584,43989,44812,0.046445


In [157]:
comparison_v3a2 = v3a1_results.merge(
    v3a2_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v3a1",
        "_v3a2",
    ),
)

In [158]:
comparison_v3a2["ap_change"] = (
    comparison_v3a2["average_precision_v3a2"]
    - comparison_v3a2["average_precision_v3a1"]
)

comparison_v3a2["ap_change_pct"] = (
    comparison_v3a2["ap_change"] / comparison_v3a2["average_precision_v3a1"] * 100
)

comparison_v3a2["fp_change"] = (
    comparison_v3a2["false_positives_at_recall_floor_v3a2"]
    - comparison_v3a2["false_positives_at_recall_floor_v3a1"]
)

comparison_v3a2["alert_rate_change"] = (
    comparison_v3a2["alert_rate_at_recall_floor_v3a2"]
    - comparison_v3a2["alert_rate_at_recall_floor_v3a1"]
)

In [159]:
comparison_v3a2[
    [
        "fold",
        "average_precision_v3a1",
        "average_precision_v3a2",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v3a1",
        "precision_at_recall_floor_v3a2",
        "false_positives_at_recall_floor_v3a1",
        "false_positives_at_recall_floor_v3a2",
        "fp_change",
        "alert_rate_at_recall_floor_v3a1",
        "alert_rate_at_recall_floor_v3a2",
        "alert_rate_change",
    ]
]

,fold,average_precision_v3a1,average_precision_v3a2,ap_change,ap_change_pct,precision_at_recall_floor_v3a1,precision_at_recall_floor_v3a2,false_positives_at_recall_floor_v3a1,false_positives_at_recall_floor_v3a2,fp_change,alert_rate_at_recall_floor_v3a1,alert_rate_at_recall_floor_v3a2,alert_rate_change
0,fold_1,0.099760,0.086501,-0.013259,-13.291108,0.028147,0.029050,11256,10896,-360,0.055836,0.054100,-0.001736
1,fold_2,0.060973,0.053027,-0.007947,-13.033010,0.009307,0.008552,40128,43705,3577,0.083922,0.091333,0.007411
2,fold_3,0.095932,0.090941,-0.004991,-5.203138,0.019294,0.018366,41833,43989,2156,0.044210,0.046445,0.002235


### V3-A2 Decision — DROP

**Feature tested:** V3-A1 + `sender_tx_count_24h`

Adding the sender's transaction count over the previous 24 hours did not improve the existing V3-A1 feature set.

Average Precision decreased across all three chronological validation folds:

* Fold 1: **−13.29%**
* Fold 2: **−13.03%**
* Fold 3: **−5.20%**

Operational performance was also unstable. Fold 1 produced a small reduction of **360 false positives**, but Folds 2 and 3 increased false positives by **3,577** and **2,156**, respectively. Precision and alert rate also deteriorated in those later folds.

The consistent AP decline suggests that `sender_tx_count_24h` does not provide useful incremental ranking information once `sender_tx_count_1h` is already included. The broader activity count may be redundant with the stronger short-term velocity signal or may introduce less discriminative normal activity into the representation.

**Decision:** DROP `sender_tx_count_24h`. Retain V3-A1, including `sender_tx_count_1h`, as the provisional feature base.


# Experiment 11 -  V3-A1 + previous-24h sender monetary volume

In [160]:
def add_sender_amount_sum_24h(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add the sender's total amount paid during the previous
    24 hours in the same payment currency as the current
    transaction.

    Historical window:
        [t - 24 hours, t)

    Therefore:
    - current transaction is excluded
    - transactions at the exact current timestamp are excluded
    - future transactions are excluded

    Sender identity:
        from_bank_id + from_account_id

    Currency:
        payment_currency
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "transaction_timestamp",
        "payment_currency",
        "amount_paid",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    # Temporary key allows missing currencies to be handled
    # consistently without changing the original column.
    result["_payment_currency_key"] = (
        result["payment_currency"].astype("string").fillna("__MISSING__")
    )

    sender_currency_cols = [
        "from_bank_id",
        "from_account_id",
        "_payment_currency_key",
    ]

    # ---------------------------------------------------------
    # 1. Aggregate amount paid by sender + currency
    #    at each exact timestamp.
    #
    # This is important because multiple transactions may
    # occur at exactly the same timestamp.
    # ---------------------------------------------------------

    timestamp_amounts = (
        result.groupby(
            sender_currency_cols + ["transaction_timestamp"],
            dropna=False,
            as_index=False,
        )["amount_paid"]
        .sum()
        .rename(columns={"amount_paid": "amount_paid_at_timestamp"})
    )

    timestamp_amounts = timestamp_amounts.sort_values(
        sender_currency_cols + ["transaction_timestamp"]
    )

    # ---------------------------------------------------------
    # 2. Sum only strictly earlier activity over 24 hours.
    # ---------------------------------------------------------

    rolling_amounts = (
        timestamp_amounts.set_index("transaction_timestamp")
        .groupby(
            sender_currency_cols,
            dropna=False,
        )["amount_paid_at_timestamp"]
        .rolling(
            window="24h",
            closed="left",
        )
        .sum()
        .fillna(0)
        .rename("sender_amount_paid_sum_24h_same_currency")
        .reset_index()
    )

    # ---------------------------------------------------------
    # 3. Merge historical value back to each transaction.
    # ---------------------------------------------------------

    result = result.merge(
        rolling_amounts[
            sender_currency_cols
            + [
                "transaction_timestamp",
                "sender_amount_paid_sum_24h_same_currency",
            ]
        ],
        on=(sender_currency_cols + ["transaction_timestamp"]),
        how="left",
        validate="many_to_one",
        sort=False,
    )

    # Heavy-tail-safe representation for the model.
    result["log_sender_amount_paid_sum_24h_same_currency"] = np.log1p(
        result["sender_amount_paid_sum_24h_same_currency"].clip(lower=0)
    )

    result = (
        result.sort_values("_row_order")
        .drop(
            columns=[
                "_row_order",
                "_payment_currency_key",
            ]
        )
        .reset_index(drop=True)
    )

    return result

In [161]:
v3a3_df = add_sender_amount_sum_24h(v3a1_df)

In [162]:
assert v3a3_df["sender_amount_paid_sum_24h_same_currency"].notna().all()

assert v3a3_df["log_sender_amount_paid_sum_24h_same_currency"].notna().all()

print("✅ No missing rolling amount values.")

✅ No missing rolling amount values.


In [163]:
assert (v3a3_df["sender_amount_paid_sum_24h_same_currency"] >= 0).all()

print("✅ Rolling amounts are non-negative.")

✅ Rolling amounts are non-negative.


In [164]:
history_group_cols = [
    "from_bank_id",
    "from_account_id",
    "payment_currency",
]

first_currency_timestamp = v3a3_df.groupby(
    history_group_cols,
    dropna=False,
)["transaction_timestamp"].transform("min")

first_currency_timestamp_mask = (
    v3a3_df["transaction_timestamp"] == first_currency_timestamp
)

assert (
    v3a3_df.loc[
        first_currency_timestamp_mask,
        "sender_amount_paid_sum_24h_same_currency",
    ]
    == 0
).all()

print("✅ First sender-currency timestamp leakage check passed.")

✅ First sender-currency timestamp leakage check passed.


In [165]:
v3a3_df["sender_amount_paid_sum_24h_same_currency"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.731672e+06
mean     6.055489e+08
std      4.913313e+09
min      0.000000e+00
50%      8.604250e+03
75%      2.808155e+05
90%      3.658931e+07
95%      2.083113e+09
99%      1.383530e+10
99.9%    9.092441e+10
max      1.012953e+12
Name: sender_amount_paid_sum_24h_same_currency, dtype: float64

In [166]:
v3a3_df["log_sender_amount_paid_sum_24h_same_currency"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.731672e+06
mean     8.771445e+00
std      6.447915e+00
min      0.000000e+00
50%      9.060128e+00
75%      1.254546e+01
90%      1.741527e+01
95%      2.145713e+01
99%      2.335049e+01
99.9%    2.523329e+01
max      2.764389e+01
Name: log_sender_amount_paid_sum_24h_same_currency, dtype: float64

In [167]:
print(
    "Zero-history transactions:",
    (v3a3_df["sender_amount_paid_sum_24h_same_currency"] == 0).sum(),
)

print(
    "Maximum raw 24h sender amount:",
    v3a3_df["sender_amount_paid_sum_24h_same_currency"].max(),
)

Zero-history transactions: 793156
Maximum raw 24h sender amount: 1012952828108.6299


## V3-A3

In [168]:
V3A3_NUMERIC_FEATURES = V3A1_NUMERIC_FEATURES + [
    "log_sender_amount_paid_sum_24h_same_currency"
]

V3A3_CATEGORICAL_FEATURES = V3A1_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V3A3 = V3A3_NUMERIC_FEATURES + V3A3_CATEGORICAL_FEATURES

In [169]:
print(
    "Added:",
    set(FEATURE_COLUMNS_V3A3) - set(FEATURE_COLUMNS_V3A1),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V3A1) - set(FEATURE_COLUMNS_V3A3),
)

Added: {'log_sender_amount_paid_sum_24h_same_currency'}
Removed: set()


In [170]:
assert set(FEATURE_COLUMNS_V3A3) - set(FEATURE_COLUMNS_V3A1) == {
    "log_sender_amount_paid_sum_24h_same_currency"
}

assert set(FEATURE_COLUMNS_V3A1) - set(FEATURE_COLUMNS_V3A3) == set()

In [171]:
splits = build_walkforward_splits(v3a3_df)

In [172]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )

fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [173]:
v3a3_fold_results = []
v3a3_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v3a3_df.iloc[train_idx][FEATURE_COLUMNS_V3A3]
    y_train = v3a3_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v3a3_df.iloc[val_idx][FEATURE_COLUMNS_V3A3]
    y_val = v3a3_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_lightgbm_pipeline(
        make_preprocessor, V3A3_NUMERIC_FEATURES, V3A3_CATEGORICAL_FEATURES
    )

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v3a3_fold_results.append(metrics)

    v3a3_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1


Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.111973
PR lift   : 57.07x
ROC-AUC   : 0.967631
Recall    : 80.0983%
Precision : 3.0399%
FP        : 10,398
Alerts    : 10,724
Alert rate: 5.1699%
Fit time  : 17.31s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.075083
PR lift   : 76.94x
ROC-AUC   : 0.963467
Recall    : 80.0425%
Precision : 1.0131%
FP        : 36,836
Alerts    : 37,213
Alert rate: 7.7101%
Fit time  : 24.31s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.118588
PR lift   : 111.30x
ROC-AUC   : 0.974084
Recall    : 80.0584%
Precision : 2.3508%
FP        : 34,186
Alerts    : 35,009
Alert rate: 3.6285%
Fit time  : 39.32s


In [174]:
v3a3_results = build_fold_metrics_table(v3a3_fold_results)

v3a3_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.111973,57.067623,0.967631,0.030399,0.800983,10398,10724,0.051699
1,fold_2,0.075083,76.940524,0.963467,0.010131,0.800425,36836,37213,0.077101
2,fold_3,0.118588,111.302073,0.974084,0.023508,0.800584,34186,35009,0.036285


In [175]:
comparison_v3a3 = v3a1_results.merge(
    v3a3_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v3a1",
        "_v3a3",
    ),
)

In [176]:
comparison_v3a3["ap_change"] = (
    comparison_v3a3["average_precision_v3a3"]
    - comparison_v3a3["average_precision_v3a1"]
)

comparison_v3a3["ap_change_pct"] = (
    comparison_v3a3["ap_change"] / comparison_v3a3["average_precision_v3a1"] * 100
)

comparison_v3a3["fp_change"] = (
    comparison_v3a3["false_positives_at_recall_floor_v3a3"]
    - comparison_v3a3["false_positives_at_recall_floor_v3a1"]
)

comparison_v3a3["alert_rate_change"] = (
    comparison_v3a3["alert_rate_at_recall_floor_v3a3"]
    - comparison_v3a3["alert_rate_at_recall_floor_v3a1"]
)

In [177]:
comparison_v3a3[
    [
        "fold",
        "average_precision_v3a1",
        "average_precision_v3a3",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v3a1",
        "precision_at_recall_floor_v3a3",
        "false_positives_at_recall_floor_v3a1",
        "false_positives_at_recall_floor_v3a3",
        "fp_change",
        "alert_rate_at_recall_floor_v3a1",
        "alert_rate_at_recall_floor_v3a3",
        "alert_rate_change",
    ]
]

,fold,average_precision_v3a1,average_precision_v3a3,ap_change,ap_change_pct,precision_at_recall_floor_v3a1,precision_at_recall_floor_v3a3,false_positives_at_recall_floor_v3a1,false_positives_at_recall_floor_v3a3,fp_change,alert_rate_at_recall_floor_v3a1,alert_rate_at_recall_floor_v3a3,alert_rate_change
0,fold_1,0.099760,0.111973,0.012212,12.241679,0.028147,0.030399,11256,10398,-858,0.055836,0.051699,-0.004136
1,fold_2,0.060973,0.075083,0.014110,23.141470,0.009307,0.010131,40128,36836,-3292,0.083922,0.077101,-0.006821
2,fold_3,0.095932,0.118588,0.022656,23.616213,0.019294,0.023508,41833,34186,-7647,0.044210,0.036285,-0.007926


## V3-A3 Decision — KEEP

**Feature tested:** V3-A1 + `log_sender_amount_paid_sum_24h_same_currency`

Adding the sender's previous-24-hour monetary volume produced a consistent improvement across all three chronological validation folds.

Average Precision improved by:

* Fold 1: **+12.24%**
* Fold 2: **+23.14%**
* Fold 3: **+23.62%**

Operational performance also improved consistently at the ≥80% recall operating point.

False positives decreased by:

* Fold 1: **−858**
* Fold 2: **−3,292**
* Fold 3: **−7,647**

Precision increased in all three folds, while alert rate decreased across every validation period.

The result suggests that recent sender monetary volume provides complementary behavioral information beyond short-term transaction count alone. `sender_tx_count_1h` captures immediate transaction velocity, while the 24-hour same-currency amount feature captures broader monetary intensity.

**Decision:** KEEP `log_sender_amount_paid_sum_24h_same_currency` and promote V3-A3 as the new provisional feature base.


# Experiment 12 - V3-A3 + sender recency(log)

In [222]:
def add_sender_recency(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add time since the sender's previous transaction.

    Only transactions at strictly earlier timestamps are used.
    Transactions occurring at the exact same timestamp do not
    count one another as previous activity.

    Sender identity:
        from_bank_id + from_account_id

    Model feature:
        log_sender_hours_since_prev_tx

    Encoding:
        -1 = no earlier sender transaction observed
        >= 0 = log1p(hours since previous transaction)
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    sender_cols = [
        "from_bank_id",
        "from_account_id",
    ]

    # ---------------------------------------------------------
    # 1. Build one row per sender + unique timestamp.
    #
    # This prevents transactions occurring at the same exact
    # timestamp from treating one another as prior activity.
    # ---------------------------------------------------------

    sender_timestamps = (
        result[sender_cols + ["transaction_timestamp"]]
        .drop_duplicates()
        .sort_values(sender_cols + ["transaction_timestamp"])
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # 2. Find the strictly previous timestamp for each sender.
    # ---------------------------------------------------------

    sender_timestamps["previous_sender_timestamp"] = sender_timestamps.groupby(
        sender_cols,
        sort=False,
    )["transaction_timestamp"].shift(1)

    # ---------------------------------------------------------
    # 3. Calculate hours since that previous transaction.
    # ---------------------------------------------------------

    sender_timestamps["sender_hours_since_prev_tx"] = (
        sender_timestamps["transaction_timestamp"]
        - sender_timestamps["previous_sender_timestamp"]
    ).dt.total_seconds() / 3600.0

    # ---------------------------------------------------------
    # 4. Log-transform recency.
    #
    # First observed sender transaction gets -1.
    # All actual recencies are >= 0.
    # ---------------------------------------------------------

    has_previous = sender_timestamps["previous_sender_timestamp"].notna()

    sender_timestamps["log_sender_hours_since_prev_tx"] = -1.0

    sender_timestamps.loc[
        has_previous,
        "log_sender_hours_since_prev_tx",
    ] = np.log1p(
        sender_timestamps.loc[
            has_previous,
            "sender_hours_since_prev_tx",
        ]
    )

    # ---------------------------------------------------------
    # 5. Merge recency back onto every original transaction.
    # ---------------------------------------------------------

    result = result.merge(
        sender_timestamps[
            sender_cols
            + [
                "transaction_timestamp",
                "sender_hours_since_prev_tx",
                "log_sender_hours_since_prev_tx",
            ]
        ],
        on=sender_cols + ["transaction_timestamp"],
        how="left",
        validate="many_to_one",
        sort=False,
    )

    result = (
        result.sort_values("_row_order")
        .drop(columns="_row_order")
        .reset_index(drop=True)
    )

    return result

In [223]:
v3b1_df = add_sender_recency(v3a3_df)

In [224]:
assert v3b1_df["log_sender_hours_since_prev_tx"].notna().all()

print("✅ No missing model recency values.")

✅ No missing model recency values.


In [225]:
known_history = v3b1_df["sender_hours_since_prev_tx"].notna()

assert (
    v3b1_df.loc[
        known_history,
        "sender_hours_since_prev_tx",
    ]
    > 0
).all()

print("✅ All known recencies are strictly positive.")

✅ All known recencies are strictly positive.


In [226]:
sender_cols = [
    "from_bank_id",
    "from_account_id",
]

first_sender_timestamp = v3b1_df.groupby(sender_cols)[
    "transaction_timestamp"
].transform("min")

first_timestamp_mask = v3b1_df["transaction_timestamp"] == first_sender_timestamp

In [227]:
assert (
    v3b1_df.loc[
        first_timestamp_mask,
        "log_sender_hours_since_prev_tx",
    ]
    == -1
).all()

print("✅ First-sender-timestamp leakage check passed.")

✅ First-sender-timestamp leakage check passed.


In [228]:
assert (
    v3b1_df.loc[
        ~first_timestamp_mask,
        "log_sender_hours_since_prev_tx",
    ]
    >= 0
).all()

print("✅ Later sender timestamps have valid history.")

✅ Later sender timestamps have valid history.


In [229]:
v3b1_df["sender_hours_since_prev_tx"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.231915e+06
mean     7.347478e+00
std      1.561244e+01
min      1.666667e-02
50%      2.833333e-01
75%      8.550000e+00
90%      2.323333e+01
95%      3.141667e+01
99%      7.838333e+01
99.9%    1.470514e+02
max      1.678667e+02
Name: sender_hours_since_prev_tx, dtype: float64

In [230]:
v3b1_df["log_sender_hours_since_prev_tx"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.731672e+06
mean     8.354354e-01
std      1.414522e+00
min     -1.000000e+00
50%      1.823216e-01
75%      1.987874e+00
90%      3.094068e+00
95%      3.405079e+00
99%      4.334017e+00
99.9%    4.983264e+00
max      5.129109e+00
Name: log_sender_hours_since_prev_tx, dtype: float64

In [231]:
print(
    "Transactions with no prior sender history:",
    (v3b1_df["log_sender_hours_since_prev_tx"] == -1).sum(),
)

Transactions with no prior sender history: 499757


In [232]:
active_sender = (
    v3b1_df.groupby(
        [
            "from_bank_id",
            "from_account_id",
        ]
    )
    .size()
    .sort_values(ascending=False)
    .index[0]
)

bank_id, account_id = active_sender

In [233]:
v3b1_df.loc[
    (v3b1_df["from_bank_id"] == bank_id) & (v3b1_df["from_account_id"] == account_id),
    [
        "transaction_timestamp",
        "sender_hours_since_prev_tx",
        "log_sender_hours_since_prev_tx",
        "sender_tx_count_1h",
    ],
].sort_values("transaction_timestamp").head(30)

,transaction_timestamp,sender_hours_since_prev_tx,log_sender_hours_since_prev_tx,sender_tx_count_1h
7,2022-09-01,NaN,-1.0,0
7470,2022-09-01,NaN,-1.0,0
7413,2022-09-01,NaN,-1.0,0
7336,2022-09-01,NaN,-1.0,0
7151,2022-09-01,NaN,-1.0,0
6999,2022-09-01,NaN,-1.0,0
6399,2022-09-01,NaN,-1.0,0
6361,2022-09-01,NaN,-1.0,0
6359,2022-09-01,NaN,-1.0,0
6325,2022-09-01,NaN,-1.0,0


## V3-B1

In [234]:
V3B1_NUMERIC_FEATURES = V3A3_NUMERIC_FEATURES + ["log_sender_hours_since_prev_tx"]

V3B1_CATEGORICAL_FEATURES = V3A3_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V3B1 = V3B1_NUMERIC_FEATURES + V3B1_CATEGORICAL_FEATURES

In [235]:
print(
    "Added:",
    set(FEATURE_COLUMNS_V3B1) - set(FEATURE_COLUMNS_V3A3),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V3A3) - set(FEATURE_COLUMNS_V3B1),
)

Added: {'log_sender_hours_since_prev_tx'}
Removed: set()


In [236]:
assert set(FEATURE_COLUMNS_V3B1) - set(FEATURE_COLUMNS_V3A3) == {
    "log_sender_hours_since_prev_tx"
}

assert set(FEATURE_COLUMNS_V3A3) - set(FEATURE_COLUMNS_V3B1) == set()

In [237]:
splits = build_walkforward_splits(v3b1_df)

In [238]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )


fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [239]:
v3b1_fold_results = []
v3b1_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v3b1_df.iloc[train_idx][FEATURE_COLUMNS_V3B1]
    y_train = v3b1_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v3b1_df.iloc[val_idx][FEATURE_COLUMNS_V3B1]
    y_val = v3b1_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_lightgbm_pipeline(
        make_preprocessor, V3B1_NUMERIC_FEATURES, V3B1_CATEGORICAL_FEATURES
    )

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v3b1_fold_results.append(metrics)

    v3b1_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1


Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.193420
PR lift   : 98.58x
ROC-AUC   : 0.969591
Recall    : 80.3440%
Precision : 2.9526%
FP        : 10,748
Alerts    : 11,075
Alert rate: 5.3392%
Fit time  : 24.76s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.134993
PR lift   : 138.33x
ROC-AUC   : 0.960457
Recall    : 80.0425%
Precision : 0.9459%
FP        : 39,480
Alerts    : 39,857
Alert rate: 8.2580%
Fit time  : 23.08s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.258269
PR lift   : 242.40x
ROC-AUC   : 0.975872
Recall    : 80.0584%
Precision : 2.5101%
FP        : 31,965
Alerts    : 32,788
Alert rate: 3.3983%
Fit time  : 23.69s


In [240]:
v3b1_results = build_fold_metrics_table(v3b1_fold_results)

v3b1_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.193420,98.577650,0.969591,0.029526,0.803440,10748,11075,0.053392
1,fold_2,0.134993,138.332499,0.960457,0.009459,0.800425,39480,39857,0.082580
2,fold_3,0.258269,242.401116,0.975872,0.025101,0.800584,31965,32788,0.033983


In [241]:
comparison_v3b1 = v3a3_results.merge(
    v3b1_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v3a3",
        "_v3b1",
    ),
)

In [242]:
comparison_v3b1["ap_change"] = (
    comparison_v3b1["average_precision_v3b1"]
    - comparison_v3b1["average_precision_v3a3"]
)

comparison_v3b1["ap_change_pct"] = (
    comparison_v3b1["ap_change"] / comparison_v3b1["average_precision_v3a3"] * 100
)

comparison_v3b1["fp_change"] = (
    comparison_v3b1["false_positives_at_recall_floor_v3b1"]
    - comparison_v3b1["false_positives_at_recall_floor_v3a3"]
)

comparison_v3b1["alert_rate_change"] = (
    comparison_v3b1["alert_rate_at_recall_floor_v3b1"]
    - comparison_v3b1["alert_rate_at_recall_floor_v3a3"]
)

In [243]:
comparison_v3b1["ap_change"] = (
    comparison_v3b1["average_precision_v3b1"]
    - comparison_v3b1["average_precision_v3a3"]
)

comparison_v3b1["ap_change_pct"] = (
    comparison_v3b1["ap_change"] / comparison_v3b1["average_precision_v3a3"] * 100
)

comparison_v3b1["fp_change"] = (
    comparison_v3b1["false_positives_at_recall_floor_v3b1"]
    - comparison_v3b1["false_positives_at_recall_floor_v3a3"]
)

comparison_v3b1["alert_rate_change"] = (
    comparison_v3b1["alert_rate_at_recall_floor_v3b1"]
    - comparison_v3b1["alert_rate_at_recall_floor_v3a3"]
)

In [244]:
comparison_v3b1[
    [
        "fold",
        "average_precision_v3a3",
        "average_precision_v3b1",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v3a3",
        "precision_at_recall_floor_v3b1",
        "false_positives_at_recall_floor_v3a3",
        "false_positives_at_recall_floor_v3b1",
        "fp_change",
        "alert_rate_at_recall_floor_v3a3",
        "alert_rate_at_recall_floor_v3b1",
        "alert_rate_change",
    ]
]

,fold,average_precision_v3a3,average_precision_v3b1,ap_change,ap_change_pct,precision_at_recall_floor_v3a3,precision_at_recall_floor_v3b1,false_positives_at_recall_floor_v3a3,false_positives_at_recall_floor_v3b1,fp_change,alert_rate_at_recall_floor_v3a3,alert_rate_at_recall_floor_v3b1,alert_rate_change
0,fold_1,0.111973,0.193420,0.081447,72.738315,0.030399,0.029526,10398,10748,350,0.051699,0.053392,0.001692
1,fold_2,0.075083,0.134993,0.059910,79.791469,0.010131,0.009459,36836,39480,2644,0.077101,0.082580,0.005478
2,fold_3,0.118588,0.258269,0.139681,117.786703,0.023508,0.025101,34186,31965,-2221,0.036285,0.033983,-0.002302


## V3-B1 Decision — KEEP

**Feature tested:** V3-A3 + `log_sender_hours_since_prev_tx`

Adding sender recency produced a large and temporally consistent improvement in fraud-ranking performance across all three chronological validation folds.

Average Precision improved by:

* Fold 1: **+72.74%**
* Fold 2: **+79.79%**
* Fold 3: **+117.79%**

This is one of the strongest feature improvements observed so far and indicates that the time elapsed since the sender's previous transaction contains substantial behavioral information beyond short-term transaction count and recent monetary volume.

Operational performance at the ≥80% recall point was mixed. False positives increased slightly in Fold 1 (**+350**) and more noticeably in Fold 2 (**+2,644**), while Fold 3 improved substantially with **2,221 fewer false positives**, higher precision, and a lower alert rate.

Therefore, the feature does not uniformly improve the specific ≥80% recall operating point, but its ranking improvement is large and consistent across every chronological fold. Since PR-AUC is the primary model-selection metric and final threshold selection will be performed separately using chronological out-of-fold predictions, the temporary operating-point tradeoff is not sufficient reason to discard the feature.

**Decision:** KEEP `log_sender_hours_since_prev_tx` and promote V3-B1 as the new provisional feature base.


# Experiment 13 - V3B1 + account relative amount
is the current payment unusually large or small compared with what this sender historically pays?

In [245]:
def add_sender_relative_amount(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add the current amount paid relative to the sender's
    historical median amount paid in the same payment currency.

    Historical information uses only transactions occurring at
    strictly earlier timestamps.

    Sender identity:
        from_bank_id + from_account_id

    Historical comparison:
        same payment_currency

    Model feature:
        log_amount_paid_vs_sender_median
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "transaction_timestamp",
        "payment_currency",
        "amount_paid",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    # Avoid grouping/merge ambiguity for missing currencies.
    result["_payment_currency_key"] = (
        result["payment_currency"].astype("string").fillna("__MISSING__")
    )

    group_cols = [
        "from_bank_id",
        "from_account_id",
        "_payment_currency_key",
    ]

    sort_cols = group_cols + ["transaction_timestamp"]

    # Stable deterministic ordering.
    if "transaction_id" in result.columns:
        sort_cols = sort_cols + ["transaction_id"]

    result = result.sort_values(
        sort_cols,
        kind="mergesort",
    ).reset_index(drop=True)

    # ---------------------------------------------------------
    # 1. Shift by one transaction.
    #
    # For the FIRST transaction at any timestamp, this creates
    # a history containing only earlier rows.
    # ---------------------------------------------------------

    result["_previous_amount"] = result.groupby(
        group_cols,
        sort=False,
        dropna=False,
    )["amount_paid"].shift(1)

    # ---------------------------------------------------------
    # 2. Expanding median of previous amounts.
    #
    # At the first row of each timestamp this represents the
    # exact median of all strictly earlier transactions.
    # ---------------------------------------------------------

    result["_preliminary_historical_median"] = (
        result.groupby(
            group_cols,
            sort=False,
            dropna=False,
        )["_previous_amount"]
        .expanding()
        .median()
        .reset_index(
            level=list(range(len(group_cols))),
            drop=True,
        )
    )

    timestamp_cols = group_cols + ["transaction_timestamp"]

    # ---------------------------------------------------------
    # 3. Keep ONLY the first row at each exact timestamp.
    #
    # Why?
    #
    # Later rows at the same timestamp would otherwise start
    # seeing other transactions at that timestamp.
    #
    # The first row's historical median is guaranteed to use
    # strictly earlier timestamps only.
    # ---------------------------------------------------------

    timestamp_history = (
        result.groupby(
            timestamp_cols,
            sort=False,
            dropna=False,
        )
        .head(1)[timestamp_cols + ["_preliminary_historical_median"]]
        .rename(
            columns={
                "_preliminary_historical_median": "sender_historical_median_amount_paid"
            }
        )
    )

    # Drop preliminary version before merging the safe value.
    result = result.drop(
        columns=[
            "_previous_amount",
            "_preliminary_historical_median",
        ]
    )

    result = result.merge(
        timestamp_history,
        on=timestamp_cols,
        how="left",
        validate="many_to_one",
        sort=False,
    )

    # ---------------------------------------------------------
    # 4. Create the actual model feature.
    #
    # NaN remains NaN when there is no previous history.
    # The pipeline's SimpleImputer will learn how to impute
    # those values inside each training fold.
    # ---------------------------------------------------------

    result["log_amount_paid_vs_sender_median"] = np.log1p(
        result["amount_paid"]
    ) - np.log1p(result["sender_historical_median_amount_paid"])

    # Restore original row order.
    result = (
        result.sort_values("_row_order")
        .drop(
            columns=[
                "_row_order",
                "_payment_currency_key",
            ]
        )
        .reset_index(drop=True)
    )

    return result

In [246]:
v3c1_df = add_sender_relative_amount(v3b1_df)

## Leakage check #1 — first sender/currency timestamp

In [247]:
history_group_cols = [
    "from_bank_id",
    "from_account_id",
    "payment_currency",
]

first_timestamp = v3c1_df.groupby(
    history_group_cols,
    dropna=False,
)["transaction_timestamp"].transform("min")

first_timestamp_mask = v3c1_df["transaction_timestamp"] == first_timestamp

assert (
    v3c1_df.loc[
        first_timestamp_mask,
        "sender_historical_median_amount_paid",
    ]
    .isna()
    .all()
)

print("✅ First sender-currency timestamp has no historical median.")

✅ First sender-currency timestamp has no historical median.


## Leakage check #2 — same timestamp gets same history

In [248]:
same_timestamp_max_unique = (
    v3c1_df.groupby(
        [
            "from_bank_id",
            "from_account_id",
            "payment_currency",
            "transaction_timestamp",
        ],
        dropna=False,
    )["sender_historical_median_amount_paid"]
    .nunique(dropna=False)
    .max()
)

assert same_timestamp_max_unique == 1

print("✅ Same-timestamp transactions share the same past-only history.")

✅ Same-timestamp transactions share the same past-only history.


In [249]:
known_history = v3c1_df["sender_historical_median_amount_paid"].notna()

assert (
    v3c1_df.loc[
        known_history,
        "sender_historical_median_amount_paid",
    ]
    >= 0
).all()

print("✅ Historical medians are non-negative.")

✅ Historical medians are non-negative.


In [250]:
assert (
    v3c1_df.loc[
        known_history,
        "log_amount_paid_vs_sender_median",
    ]
    .notna()
    .all()
)

print("✅ Relative amount values are valid.")

✅ Relative amount values are valid.


In [251]:
v3c1_df["sender_historical_median_amount_paid"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.225437e+06
mean     2.246350e+06
std      4.225925e+08
min      1.000000e-06
50%      1.623600e+03
75%      9.837505e+03
90%      9.244850e+04
95%      3.138548e+05
99%      5.228287e+06
99.9%    1.793459e+08
max      5.064764e+11
Name: sender_historical_median_amount_paid, dtype: float64

In [252]:
v3c1_df["log_amount_paid_vs_sender_median"].describe(
    percentiles=[
        0.01,
        0.05,
        0.25,
        0.50,
        0.75,
        0.95,
        0.99,
    ]
)

count    3.225437e+06
mean    -1.245804e-01
std      2.873341e+00
min     -1.973844e+01
1%      -8.171155e+00
5%      -4.785094e+00
25%     -1.419342e+00
50%      0.000000e+00
75%      9.936554e-01
95%      4.868753e+00
99%      8.630515e+00
max      2.108953e+01
Name: log_amount_paid_vs_sender_median, dtype: float64

In [253]:
print(
    "Transactions without sender/currency history:",
    v3c1_df["sender_historical_median_amount_paid"].isna().sum(),
)

Transactions without sender/currency history: 506235


In [254]:
v3c1_df.loc[
    v3c1_df["sender_historical_median_amount_paid"].notna(),
    [
        "transaction_timestamp",
        "from_bank_id",
        "from_account_id",
        "payment_currency",
        "amount_paid",
        "sender_historical_median_amount_paid",
        "log_amount_paid_vs_sender_median",
        "is_laundering",
    ],
].sort_values(
    "log_amount_paid_vs_sender_median",
    ascending=False,
).head(20)

,transaction_timestamp,from_bank_id,from_account_id,payment_currency,amount_paid,sender_historical_median_amount_paid,log_amount_paid_vs_sender_median,is_laundering
1086837,2022-09-01 23:10:00,1502,802A0AFD0,Euro,7.009740e+09,3.86,21.089528,0
814917,2022-09-01 15:03:00,217959,812C62220,Euro,2.152456e+09,4.31,19.820283,0
365876,2022-09-01 01:39:00,20,8000FA250,Euro,1.853152e+09,6.60,19.312005,0
3021405,2022-09-06 12:17:00,1024,802BADD20,Euro,1.950997e+09,18.20,18.436696,0
3571371,2022-09-07 15:46:00,120438,80B32BEC0,US Dollar,1.535090e+09,16.72,18.277161,0
1518557,2022-09-02 12:36:00,1588,8037B1E60,US Dollar,1.538163e+09,18.90,18.163135,0
667569,2022-09-01 10:40:00,220,8006005D0,US Dollar,3.682944e+08,3.82,18.151619,0
247607,2022-09-01 00:22:00,217959,8068D89F0,Euro,7.542513e+08,10.59,17.991094,0
182446,2022-09-01 00:16:00,11056,80A1C5E20,Euro,1.358033e+09,20.94,17.940992,0
289790,2022-09-01 00:26:00,1362,800428E90,US Dollar,2.769401e+08,3.55,17.924184,0


## V3-C1

In [255]:
V3C1_NUMERIC_FEATURES = V3B1_NUMERIC_FEATURES + ["log_amount_paid_vs_sender_median"]

V3C1_CATEGORICAL_FEATURES = V3B1_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V3C1 = V3C1_NUMERIC_FEATURES + V3C1_CATEGORICAL_FEATURES

In [256]:
print(
    "Added:",
    set(FEATURE_COLUMNS_V3C1) - set(FEATURE_COLUMNS_V3B1),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V3B1) - set(FEATURE_COLUMNS_V3C1),
)

Added: {'log_amount_paid_vs_sender_median'}
Removed: set()


In [257]:
assert set(FEATURE_COLUMNS_V3C1) - set(FEATURE_COLUMNS_V3B1) == {
    "log_amount_paid_vs_sender_median"
}

assert set(FEATURE_COLUMNS_V3B1) - set(FEATURE_COLUMNS_V3C1) == set()

## Preprocessor

In [258]:
splits = build_walkforward_splits(v3c1_df)

In [259]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )


fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [260]:
v3c1_fold_results = []
v3c1_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v3c1_df.iloc[train_idx][FEATURE_COLUMNS_V3C1]
    y_train = v3c1_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v3c1_df.iloc[val_idx][FEATURE_COLUMNS_V3C1]
    y_val = v3c1_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_lightgbm_pipeline(
        make_preprocessor, V3C1_NUMERIC_FEATURES, V3C1_CATEGORICAL_FEATURES
    )

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v3c1_fold_results.append(metrics)

    v3c1_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1


Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.322554
PR lift   : 164.39x
ROC-AUC   : 0.975786
Recall    : 80.0983%
Precision : 3.9995%
FP        : 7,825
Alerts    : 8,151
Alert rate: 3.9295%
Fit time  : 19.56s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.246479
PR lift   : 252.58x
ROC-AUC   : 0.967195
Recall    : 80.0425%
Precision : 1.0779%
FP        : 34,600
Alerts    : 34,977
Alert rate: 7.2469%
Fit time  : 19.84s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.388159
PR lift   : 364.31x
ROC-AUC   : 0.979167
Recall    : 80.0584%
Precision : 3.0315%
FP        : 26,325
Alerts    : 27,148
Alert rate: 2.8137%
Fit time  : 24.15s


In [261]:
v3c1_results = build_fold_metrics_table(v3c1_fold_results)

v3c1_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.322554,164.391345,0.975786,0.039995,0.800983,7825,8151,0.039295
1,fold_2,0.246479,252.575575,0.967195,0.010779,0.800425,34600,34977,0.072469
2,fold_3,0.388159,364.310698,0.979167,0.030315,0.800584,26325,27148,0.028137


In [262]:
comparison_v3c1 = v3b1_results.merge(
    v3c1_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v3b1",
        "_v3c1",
    ),
)

In [263]:
comparison_v3c1["ap_change"] = (
    comparison_v3c1["average_precision_v3c1"]
    - comparison_v3c1["average_precision_v3b1"]
)

comparison_v3c1["ap_change_pct"] = (
    comparison_v3c1["ap_change"] / comparison_v3c1["average_precision_v3b1"] * 100
)

comparison_v3c1["fp_change"] = (
    comparison_v3c1["false_positives_at_recall_floor_v3c1"]
    - comparison_v3c1["false_positives_at_recall_floor_v3b1"]
)

comparison_v3c1["alert_rate_change"] = (
    comparison_v3c1["alert_rate_at_recall_floor_v3c1"]
    - comparison_v3c1["alert_rate_at_recall_floor_v3b1"]
)

In [264]:
comparison_v3c1[
    [
        "fold",
        "average_precision_v3b1",
        "average_precision_v3c1",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v3b1",
        "precision_at_recall_floor_v3c1",
        "false_positives_at_recall_floor_v3b1",
        "false_positives_at_recall_floor_v3c1",
        "fp_change",
        "alert_rate_at_recall_floor_v3b1",
        "alert_rate_at_recall_floor_v3c1",
        "alert_rate_change",
    ]
]

,fold,average_precision_v3b1,average_precision_v3c1,ap_change,ap_change_pct,precision_at_recall_floor_v3b1,precision_at_recall_floor_v3c1,false_positives_at_recall_floor_v3b1,false_positives_at_recall_floor_v3c1,fp_change,alert_rate_at_recall_floor_v3b1,alert_rate_at_recall_floor_v3c1,alert_rate_change
0,fold_1,0.193420,0.322554,0.129134,66.763304,0.029526,0.039995,10748,7825,-2923,0.053392,0.039295,-0.014096
1,fold_2,0.134993,0.246479,0.111486,82.585855,0.009459,0.010779,39480,34600,-4880,0.082580,0.072469,-0.010111
2,fold_3,0.258269,0.388159,0.129890,50.292500,0.025101,0.030315,31965,26325,-5640,0.033983,0.028137,-0.005846


## V3-C1 Decision — KEEP

**Feature tested:** V3-B1 + `log_amount_paid_vs_sender_median`

Adding the current transaction amount relative to the sender's historical same-currency median produced a large and consistent improvement across all three chronological validation folds.

Average Precision improved by:

* Fold 1: **+66.76%**
* Fold 2: **+82.59%**
* Fold 3: **+50.29%**

Operational performance also improved across every fold at the ≥80% recall operating point.

False positives decreased by:

* Fold 1: **−2,923**
* Fold 2: **−4,880**
* Fold 3: **−5,640**

Precision increased and alert rate decreased consistently across all three validation periods.

The result suggests that transaction magnitude becomes substantially more informative when interpreted relative to the sender's own historical behavior rather than only as an absolute amount. This helps distinguish genuinely unusual transfers from large transactions that may be normal for a particular sender.

**Decision:** KEEP `log_amount_paid_vs_sender_median` and promote V3-C1 as the new provisional feature base.


# Experiment 14 - V3-C1 + receiver_seen_before

In [265]:
def add_receiver_seen_before(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Indicate whether the sender has previously transacted
    with the same receiver.

    Only strictly earlier timestamps count as history.

    Sender:
        from_bank_id + from_account_id

    Receiver:
        to_bank_id + to_account_id

    Feature:
        receiver_seen_before

    Values:
        0 = no earlier sender-receiver transaction observed
        1 = sender has previously transacted with receiver
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    relationship_cols = [
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
    ]

    # ---------------------------------------------------------
    # One row per sender-receiver relationship + timestamp.
    #
    # This makes same-timestamp transactions share the same
    # historical state.
    # ---------------------------------------------------------

    relationship_timestamps = (
        result[relationship_cols + ["transaction_timestamp"]]
        .drop_duplicates()
        .sort_values(relationship_cols + ["transaction_timestamp"])
        .reset_index(drop=True)
    )

    # ---------------------------------------------------------
    # cumcount = 0 for first observed timestamp
    #            1+ for later timestamps.
    #
    # Therefore later relationship timestamps mean this
    # receiver was already known to this sender.
    # ---------------------------------------------------------

    relationship_timestamps["receiver_seen_before"] = (
        relationship_timestamps.groupby(
            relationship_cols,
            sort=False,
            dropna=False,
        )
        .cumcount()
        .gt(0)
        .astype("int8")
    )

    # ---------------------------------------------------------
    # Merge back to the original transactions.
    # ---------------------------------------------------------

    result = result.merge(
        relationship_timestamps[
            relationship_cols
            + [
                "transaction_timestamp",
                "receiver_seen_before",
            ]
        ],
        on=(relationship_cols + ["transaction_timestamp"]),
        how="left",
        validate="many_to_one",
        sort=False,
    )

    result = (
        result.sort_values("_row_order")
        .drop(columns="_row_order")
        .reset_index(drop=True)
    )

    return result

In [266]:
v3d1_df = add_receiver_seen_before(v3c1_df)

In [267]:
assert v3d1_df["receiver_seen_before"].notna().all()

assert set(v3d1_df["receiver_seen_before"].unique()).issubset({0, 1})

print("✅ receiver_seen_before contains only 0/1.")

✅ receiver_seen_before contains only 0/1.


In [268]:
relationship_cols = [
    "from_bank_id",
    "from_account_id",
    "to_bank_id",
    "to_account_id",
]

first_relationship_timestamp = v3d1_df.groupby(
    relationship_cols,
    dropna=False,
)["transaction_timestamp"].transform("min")

first_relationship_mask = (
    v3d1_df["transaction_timestamp"] == first_relationship_timestamp
)

assert (
    v3d1_df.loc[
        first_relationship_mask,
        "receiver_seen_before",
    ]
    == 0
).all()

print("✅ First sender-receiver timestamps are correctly marked unseen.")

✅ First sender-receiver timestamps are correctly marked unseen.


In [269]:
assert (
    v3d1_df.loc[
        ~first_relationship_mask,
        "receiver_seen_before",
    ]
    == 1
).all()

print("✅ Later sender-receiver timestamps are correctly marked seen.")

✅ Later sender-receiver timestamps are correctly marked seen.


In [270]:
same_timestamp_max_unique = (
    v3d1_df.groupby(
        relationship_cols + ["transaction_timestamp"],
        dropna=False,
    )["receiver_seen_before"]
    .nunique(dropna=False)
    .max()
)

assert same_timestamp_max_unique == 1

print("✅ Same-timestamp transactions share the same receiver history.")

✅ Same-timestamp transactions share the same receiver history.


In [271]:
v3d1_df["receiver_seen_before"].value_counts(normalize=False)

receiver_seen_before
1    2759442
0     972230
Name: count, dtype: int64

In [272]:
v3d1_df["receiver_seen_before"].value_counts(normalize=True)

receiver_seen_before
1    0.739465
0    0.260535
Name: proportion, dtype: float64

In [273]:
v3d1_df.groupby("receiver_seen_before")["is_laundering"].agg(
    transaction_count="size",
    fraud_count="sum",
    fraud_rate="mean",
)

,transaction_count,fraud_count,fraud_rate
receiver_seen_before,,,
0,972230,2434,0.002504
1,2759442,593,0.000215


## V3 - D1

In [275]:
V3D1_NUMERIC_FEATURES = V3C1_NUMERIC_FEATURES + ["receiver_seen_before"]

V3D1_CATEGORICAL_FEATURES = V3C1_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V3D1 = V3D1_NUMERIC_FEATURES + V3D1_CATEGORICAL_FEATURES

In [276]:
print(
    "Added:",
    set(FEATURE_COLUMNS_V3D1) - set(FEATURE_COLUMNS_V3C1),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V3C1) - set(FEATURE_COLUMNS_V3D1),
)

Added: {'receiver_seen_before'}
Removed: set()


In [277]:
assert set(FEATURE_COLUMNS_V3D1) - set(FEATURE_COLUMNS_V3C1) == {"receiver_seen_before"}

assert set(FEATURE_COLUMNS_V3C1) - set(FEATURE_COLUMNS_V3D1) == set()

In [278]:
V3D1_NUMERIC_FEATURES
V3D1_CATEGORICAL_FEATURES

['receiving_currency', 'payment_currency', 'payment_format']

In [279]:
splits = build_walkforward_splits(v3d1_df)

In [280]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )


fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [281]:
v3d1_fold_results = []
v3d1_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v3d1_df.iloc[train_idx][FEATURE_COLUMNS_V3D1]
    y_train = v3d1_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v3d1_df.iloc[val_idx][FEATURE_COLUMNS_V3D1]
    y_val = v3d1_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_lightgbm_pipeline(
        make_preprocessor, V3D1_NUMERIC_FEATURES, V3D1_CATEGORICAL_FEATURES
    )

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v3d1_fold_results.append(metrics)

    v3d1_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1
Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.368214
PR lift   : 187.66x
ROC-AUC   : 0.978236
Recall    : 80.0983%
Precision : 5.5003%
FP        : 5,601
Alerts    : 5,927
Alert rate: 2.8573%
Fit time  : 34.93s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.273613
PR lift   : 280.38x
ROC-AUC   : 0.969402
Recall    : 80.0425%
Precision : 1.1879%
FP        : 31,361
Alerts    : 31,738
Alert rate: 6.5758%
Fit time  : 29.32s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.382535
PR lift   : 359.03x
ROC-AUC   : 0.981940
Recall    : 80.0584%
Precision : 3.8088%
FP        : 20,785
Alerts    : 21,608
Alert rate: 2.2395%
Fit time  : 30.72s


In [282]:
v3d1_results = build_fold_metrics_table(v3d1_fold_results)

v3d1_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.368214,187.662273,0.978236,0.055003,0.800983,5601,5927,0.028573
1,fold_2,0.273613,280.380707,0.969402,0.011879,0.800425,31361,31738,0.065758
2,fold_3,0.382535,359.032508,0.981940,0.038088,0.800584,20785,21608,0.022395


In [283]:
comparison_v3d1 = v3c1_results.merge(
    v3d1_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v3c1",
        "_v3d1",
    ),
)

In [284]:
comparison_v3d1["ap_change"] = (
    comparison_v3d1["average_precision_v3d1"]
    - comparison_v3d1["average_precision_v3c1"]
)

comparison_v3d1["ap_change_pct"] = (
    comparison_v3d1["ap_change"] / comparison_v3d1["average_precision_v3c1"] * 100
)

comparison_v3d1["fp_change"] = (
    comparison_v3d1["false_positives_at_recall_floor_v3d1"]
    - comparison_v3d1["false_positives_at_recall_floor_v3c1"]
)

comparison_v3d1["alert_rate_change"] = (
    comparison_v3d1["alert_rate_at_recall_floor_v3d1"]
    - comparison_v3d1["alert_rate_at_recall_floor_v3c1"]
)

In [285]:
comparison_v3d1[
    [
        "fold",
        "average_precision_v3c1",
        "average_precision_v3d1",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v3c1",
        "precision_at_recall_floor_v3d1",
        "false_positives_at_recall_floor_v3c1",
        "false_positives_at_recall_floor_v3d1",
        "fp_change",
        "alert_rate_at_recall_floor_v3c1",
        "alert_rate_at_recall_floor_v3d1",
        "alert_rate_change",
    ]
]

,fold,average_precision_v3c1,average_precision_v3d1,ap_change,ap_change_pct,precision_at_recall_floor_v3c1,precision_at_recall_floor_v3d1,false_positives_at_recall_floor_v3c1,false_positives_at_recall_floor_v3d1,fp_change,alert_rate_at_recall_floor_v3c1,alert_rate_at_recall_floor_v3d1,alert_rate_change
0,fold_1,0.322554,0.368214,0.045660,14.155811,0.039995,0.055003,7825,5601,-2224,0.039295,0.028573,-0.010722
1,fold_2,0.246479,0.273613,0.027134,11.008639,0.010779,0.011879,34600,31361,-3239,0.072469,0.065758,-0.006711
2,fold_3,0.388159,0.382535,-0.005624,-1.448816,0.030315,0.038088,26325,20785,-5540,0.028137,0.022395,-0.005742


## V3-D1 Decision — KEEP

**Feature tested:** V3-C1 + `receiver_seen_before`

Adding whether the sender had previously transacted with the receiver produced useful incremental performance, particularly in reducing false-positive burden.

Average Precision changed by:

* Fold 1: **+14.16%**
* Fold 2: **+11.01%**
* Fold 3: **−1.45%**

Although Fold 3 showed a small AP decline, operational performance improved consistently across all three chronological folds.

False positives decreased by:

* Fold 1: **−2,224**
* Fold 2: **−3,239**
* Fold 3: **−5,540**

Precision at the ≥80% recall operating point increased substantially in every fold, while alert rate decreased consistently.

The small Fold 3 ranking decline is therefore outweighed by the strong and temporally consistent improvement in the quality of the alert population. The feature appears to provide useful information about counterparty novelty that is not fully captured by sender-level behavioral features.

**Decision:** KEEP `receiver_seen_before` and promote V3-D1 as the new provisional feature base.


# Experiment 15: V3-D1 + sender_receiver_prior_tx_count

In [286]:
def add_sender_receiver_prior_tx_count(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add the number of strictly earlier transactions between
    the same sender and receiver.

    Sender identity:
        from_bank_id + from_account_id

    Receiver identity:
        to_bank_id + to_account_id

    Only transactions with timestamp < current timestamp
    contribute to the count.

    Transactions occurring at the same exact timestamp do
    not count one another.
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    relationship_cols = [
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
    ]

    # ---------------------------------------------------------
    # 1. Count how many transactions occurred for each
    #    sender-receiver pair at each exact timestamp.
    # ---------------------------------------------------------

    timestamp_counts = (
        result.groupby(
            relationship_cols + ["transaction_timestamp"],
            dropna=False,
        )
        .size()
        .rename("tx_at_timestamp")
        .reset_index()
    )

    timestamp_counts = timestamp_counts.sort_values(
        relationship_cols + ["transaction_timestamp"]
    ).reset_index(drop=True)

    # ---------------------------------------------------------
    # 2. Cumulative number of relationship transactions.
    #
    # Shift first so the current timestamp is excluded.
    # ---------------------------------------------------------

    timestamp_counts["sender_receiver_prior_tx_count"] = (
        timestamp_counts.groupby(
            relationship_cols,
            sort=False,
            dropna=False,
        )["tx_at_timestamp"]
        .transform(lambda x: x.cumsum().shift(1, fill_value=0))
        .astype("int32")
    )

    # ---------------------------------------------------------
    # 3. Merge the past-only count back onto every transaction.
    # ---------------------------------------------------------

    result = result.merge(
        timestamp_counts[
            relationship_cols
            + [
                "transaction_timestamp",
                "sender_receiver_prior_tx_count",
            ]
        ],
        on=(relationship_cols + ["transaction_timestamp"]),
        how="left",
        validate="many_to_one",
        sort=False,
    )

    result = (
        result.sort_values("_row_order")
        .drop(columns="_row_order")
        .reset_index(drop=True)
    )

    return result

In [287]:
v3d2_df = add_sender_receiver_prior_tx_count(v3d1_df)

In [288]:
relationship_cols = [
    "from_bank_id",
    "from_account_id",
    "to_bank_id",
    "to_account_id",
]

first_relationship_timestamp = v3d2_df.groupby(
    relationship_cols,
    dropna=False,
)["transaction_timestamp"].transform("min")

first_relationship_mask = (
    v3d2_df["transaction_timestamp"] == first_relationship_timestamp
)

assert (
    v3d2_df.loc[
        first_relationship_mask,
        "sender_receiver_prior_tx_count",
    ]
    == 0
).all()

print("✅ First sender-receiver timestamps have zero prior transactions.")

✅ First sender-receiver timestamps have zero prior transactions.


In [289]:
assert v3d2_df["sender_receiver_prior_tx_count"].notna().all()

assert (v3d2_df["sender_receiver_prior_tx_count"] >= 0).all()

print("✅ Prior relationship counts are valid.")

✅ Prior relationship counts are valid.


In [290]:
assert (
    (v3d2_df["receiver_seen_before"] == 0)
    == (v3d2_df["sender_receiver_prior_tx_count"] == 0)
).all()

assert (
    (v3d2_df["receiver_seen_before"] == 1)
    == (v3d2_df["sender_receiver_prior_tx_count"] > 0)
).all()

print("✅ Prior count agrees perfectly with receiver_seen_before.")

✅ Prior count agrees perfectly with receiver_seen_before.


In [291]:
same_timestamp_max_unique = (
    v3d2_df.groupby(
        relationship_cols + ["transaction_timestamp"],
        dropna=False,
    )["sender_receiver_prior_tx_count"]
    .nunique(dropna=False)
    .max()
)

assert same_timestamp_max_unique == 1

print("✅ Same-timestamp relationship transactions share the same prior count.")

✅ Same-timestamp relationship transactions share the same prior count.


In [292]:
v3d2_df["sender_receiver_prior_tx_count"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.731672e+06
mean     4.945080e+00
std      5.443960e+00
min      0.000000e+00
50%      3.000000e+00
75%      8.000000e+00
90%      1.200000e+01
95%      1.600000e+01
99%      2.300000e+01
99.9%    3.100000e+01
max      6.300000e+01
Name: sender_receiver_prior_tx_count, dtype: float64

In [293]:
print(
    "Never-seen relationships:",
    (v3d2_df["sender_receiver_prior_tx_count"] == 0).sum(),
)

print(
    "Maximum prior relationship count:",
    v3d2_df["sender_receiver_prior_tx_count"].max(),
)

Never-seen relationships: 972230
Maximum prior relationship count: 63


In [294]:
v3d2_df["sender_receiver_prior_tx_count"].value_counts().head(20)

sender_receiver_prior_tx_count
0     972230
1     462525
2     266708
3     226798
4     221879
6     197620
5     186633
8     180365
9     173650
7     172062
12    103991
10    103506
13     98341
11     96768
14     47618
15     29419
16     29366
18     28697
17     27517
19     26988
Name: count, dtype: int64

## V3-D2

In [295]:
V3D2_NUMERIC_FEATURES = V3D1_NUMERIC_FEATURES + ["sender_receiver_prior_tx_count"]

V3D2_CATEGORICAL_FEATURES = V3D1_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V3D2 = V3D2_NUMERIC_FEATURES + V3D2_CATEGORICAL_FEATURES

In [296]:
print(
    "Added:",
    set(FEATURE_COLUMNS_V3D2) - set(FEATURE_COLUMNS_V3D1),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V3D1) - set(FEATURE_COLUMNS_V3D2),
)

Added: {'sender_receiver_prior_tx_count'}
Removed: set()


In [297]:
assert set(FEATURE_COLUMNS_V3D2) - set(FEATURE_COLUMNS_V3D1) == {
    "sender_receiver_prior_tx_count"
}

assert set(FEATURE_COLUMNS_V3D1) - set(FEATURE_COLUMNS_V3D2) == set()

In [298]:
V3D2_NUMERIC_FEATURES
V3D2_CATEGORICAL_FEATURES

['receiving_currency', 'payment_currency', 'payment_format']

## Preprocessor

In [299]:
splits = build_walkforward_splits(v3d2_df)

In [300]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )


fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [301]:
v3d2_fold_results = []
v3d2_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v3d2_df.iloc[train_idx][FEATURE_COLUMNS_V3D2]
    y_train = v3d2_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v3d2_df.iloc[val_idx][FEATURE_COLUMNS_V3D2]
    y_val = v3d2_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_lightgbm_pipeline(
        make_preprocessor, V3D2_NUMERIC_FEATURES, V3D2_CATEGORICAL_FEATURES
    )

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v3d2_fold_results.append(metrics)

    v3d2_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1
Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.373212
PR lift   : 190.21x
ROC-AUC   : 0.981916
Recall    : 80.0983%
Precision : 6.2464%
FP        : 4,893
Alerts    : 5,219
Alert rate: 2.5160%
Fit time  : 21.39s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.300583
PR lift   : 308.02x
ROC-AUC   : 0.974957
Recall    : 80.0425%
Precision : 1.4368%
FP        : 25,861
Alerts    : 26,238
Alert rate: 5.4362%
Fit time  : 22.66s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.393528
PR lift   : 369.35x
ROC-AUC   : 0.983764
Recall    : 80.0584%
Precision : 4.7696%
FP        : 16,432
Alerts    : 17,255
Alert rate: 1.7884%
Fit time  : 27.61s


In [302]:
v3d2_results = build_fold_metrics_table(v3d2_fold_results)

v3d2_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.373212,190.209822,0.981916,0.062464,0.800983,4893,5219,0.025160
1,fold_2,0.300583,308.017949,0.974957,0.014368,0.800425,25861,26238,0.054362
2,fold_3,0.393528,369.350184,0.983764,0.047696,0.800584,16432,17255,0.017884


In [303]:
comparison_v3d2 = v3d1_results.merge(
    v3d2_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v3d1",
        "_v3d2",
    ),
)

In [304]:
comparison_v3d2["ap_change"] = (
    comparison_v3d2["average_precision_v3d2"]
    - comparison_v3d2["average_precision_v3d1"]
)

comparison_v3d2["ap_change_pct"] = (
    comparison_v3d2["ap_change"] / comparison_v3d2["average_precision_v3d1"] * 100
)

comparison_v3d2["fp_change"] = (
    comparison_v3d2["false_positives_at_recall_floor_v3d2"]
    - comparison_v3d2["false_positives_at_recall_floor_v3d1"]
)

comparison_v3d2["alert_rate_change"] = (
    comparison_v3d2["alert_rate_at_recall_floor_v3d2"]
    - comparison_v3d2["alert_rate_at_recall_floor_v3d1"]
)

In [305]:
comparison_v3d2[
    [
        "fold",
        "average_precision_v3d1",
        "average_precision_v3d2",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v3d1",
        "precision_at_recall_floor_v3d2",
        "false_positives_at_recall_floor_v3d1",
        "false_positives_at_recall_floor_v3d2",
        "fp_change",
        "alert_rate_at_recall_floor_v3d1",
        "alert_rate_at_recall_floor_v3d2",
        "alert_rate_change",
    ]
]

,fold,average_precision_v3d1,average_precision_v3d2,ap_change,ap_change_pct,precision_at_recall_floor_v3d1,precision_at_recall_floor_v3d2,false_positives_at_recall_floor_v3d1,false_positives_at_recall_floor_v3d2,fp_change,alert_rate_at_recall_floor_v3d1,alert_rate_at_recall_floor_v3d2,alert_rate_change
0,fold_1,0.368214,0.373212,0.004999,1.357518,0.055003,0.062464,5601,4893,-708,0.028573,0.025160,-0.003413
1,fold_2,0.273613,0.300583,0.026970,9.857041,0.011879,0.014368,31361,25861,-5500,0.065758,0.054362,-0.011395
2,fold_3,0.382535,0.393528,0.010993,2.873744,0.038088,0.047696,20785,16432,-4353,0.022395,0.017884,-0.004512


## V3-D2 Decision — KEEP

**Feature tested:** V3-D1 + `sender_receiver_prior_tx_count`

Adding the number of prior transactions between the sender and receiver produced a consistent incremental improvement over the binary `receiver_seen_before` feature.

Average Precision improved across all three chronological validation folds:

* Fold 1: **+1.36%**
* Fold 2: **+9.86%**
* Fold 3: **+2.87%**

Operational performance also improved consistently at the ≥80% recall operating point.

False positives decreased by:

* Fold 1: **−708**
* Fold 2: **−5,500**
* Fold 3: **−4,353**

Precision increased and alert rate decreased across every fold.

The result indicates that relationship depth contains useful information beyond simply identifying whether a receiver is new or previously seen. A receiver used once and a receiver used hundreds of times are both marked as previously seen by the binary novelty feature, while the prior-transaction count allows LightGBM to distinguish between weak and well-established sender-receiver relationships.

**Decision:** KEEP `sender_receiver_prior_tx_count` and promote V3-D2 as the new provisional feature base.


# Experiment 16: V3-D2 + sender_distinct_receivers_24h

In [306]:
from collections import Counter, deque


def add_sender_distinct_receivers_24h(
    df: pd.DataFrame,
) -> pd.DataFrame:
    """
    Add the number of distinct receivers used by the sender
    during the previous 24 hours.

    Window:
        [t - 24 hours, t)

    Therefore:
    - current timestamp is excluded
    - same-timestamp transactions cannot see one another
    - future transactions are excluded

    Sender:
        from_bank_id + from_account_id

    Receiver:
        to_bank_id + to_account_id
    """

    required_columns = {
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
        "transaction_timestamp",
    }

    missing = required_columns - set(df.columns)

    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    result = df.copy()

    result["_row_order"] = np.arange(len(result))

    result["transaction_timestamp"] = pd.to_datetime(result["transaction_timestamp"])

    sender_cols = [
        "from_bank_id",
        "from_account_id",
    ]

    receiver_cols = [
        "to_bank_id",
        "to_account_id",
    ]

    # ---------------------------------------------------------
    # 1. One event per sender-receiver-timestamp.
    #
    # If the same sender sends multiple transactions to the
    # same receiver at exactly the same timestamp, that
    # receiver should count only once for distinct-receiver
    # history.
    # ---------------------------------------------------------

    events = (
        result[sender_cols + receiver_cols + ["transaction_timestamp"]]
        .drop_duplicates()
        .sort_values(
            sender_cols + ["transaction_timestamp"] + receiver_cols,
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    historical_counts = []

    window_size = pd.Timedelta(hours=24)

    # ---------------------------------------------------------
    # 2. Process each sender chronologically.
    # ---------------------------------------------------------

    for sender_key, sender_events in events.groupby(
        sender_cols,
        sort=False,
        dropna=False,
    ):
        # Number of currently active timestamp-events for
        # each receiver inside the 24h window.
        active_receiver_counts = Counter()

        # Historical receiver events currently inside window.
        # Each element:
        #     (timestamp, receiver_key)
        active_window = deque()

        sender_key = sender_key if isinstance(sender_key, tuple) else (sender_key,)

        current_timestamp = None
        receivers_at_current_timestamp = []

        def process_timestamp(
            timestamp,
            receivers,
        ):
            """
            Calculate history BEFORE adding receivers from the
            current timestamp.
            """

            cutoff = timestamp - window_size

            # Remove events older than t - 24h.
            #
            # Notice '<', not '<='.
            #
            # An event exactly 24h earlier remains inside
            # [t - 24h, t).
            while active_window and active_window[0][0] < cutoff:
                old_timestamp, old_receiver = active_window.popleft()

                active_receiver_counts[old_receiver] -= 1

                if active_receiver_counts[old_receiver] == 0:
                    del active_receiver_counts[old_receiver]

            # History is recorded BEFORE current timestamp
            # receivers are added.
            historical_counts.append(
                (
                    *sender_key,
                    timestamp,
                    len(active_receiver_counts),
                )
            )

            # Now current timestamp becomes available to
            # FUTURE transactions.
            for receiver in receivers:
                active_window.append((timestamp, receiver))

                active_receiver_counts[receiver] += 1

        # -----------------------------------------------------
        # Walk through sender events, one timestamp block
        # at a time.
        # -----------------------------------------------------

        for row in sender_events.itertuples(
            index=False,
            name=None,
        ):
            # Column order in events:
            # sender cols,
            # receiver cols,
            # timestamp

            receiver = (
                row[2],
                row[3],
            )

            timestamp = row[4]

            if current_timestamp is None:
                current_timestamp = timestamp

            # We reached a new timestamp.
            if timestamp != current_timestamp:
                process_timestamp(
                    current_timestamp,
                    receivers_at_current_timestamp,
                )

                current_timestamp = timestamp
                receivers_at_current_timestamp = []

            receivers_at_current_timestamp.append(receiver)

        # Process final timestamp for this sender.
        if current_timestamp is not None:
            process_timestamp(
                current_timestamp,
                receivers_at_current_timestamp,
            )

    # ---------------------------------------------------------
    # 3. Build sender/timestamp lookup table.
    # ---------------------------------------------------------

    history_df = pd.DataFrame(
        historical_counts,
        columns=[
            "from_bank_id",
            "from_account_id",
            "transaction_timestamp",
            "sender_distinct_receivers_24h",
        ],
    )

    history_df["sender_distinct_receivers_24h"] = history_df[
        "sender_distinct_receivers_24h"
    ].astype("int32")

    # ---------------------------------------------------------
    # 4. Merge history back to every original transaction.
    # ---------------------------------------------------------

    result = result.merge(
        history_df,
        on=(sender_cols + ["transaction_timestamp"]),
        how="left",
        validate="many_to_one",
        sort=False,
    )

    result = (
        result.sort_values("_row_order")
        .drop(columns="_row_order")
        .reset_index(drop=True)
    )

    return result

In [307]:
v3e1_df = add_sender_distinct_receivers_24h(v3d2_df)

## Checks

In [308]:
sender_cols = [
    "from_bank_id",
    "from_account_id",
]

first_sender_timestamp = v3e1_df.groupby(
    sender_cols,
    dropna=False,
)["transaction_timestamp"].transform("min")

first_timestamp_mask = v3e1_df["transaction_timestamp"] == first_sender_timestamp

assert (
    v3e1_df.loc[
        first_timestamp_mask,
        "sender_distinct_receivers_24h",
    ]
    == 0
).all()

print("✅ First sender timestamps have zero previous distinct receivers.")

✅ First sender timestamps have zero previous distinct receivers.


In [309]:
same_timestamp_max_unique = (
    v3e1_df.groupby(
        sender_cols + ["transaction_timestamp"],
        dropna=False,
    )["sender_distinct_receivers_24h"]
    .nunique(dropna=False)
    .max()
)

assert same_timestamp_max_unique == 1

print("✅ Same-timestamp sender transactions share identical history.")

✅ Same-timestamp sender transactions share identical history.


In [310]:
assert v3e1_df["sender_distinct_receivers_24h"].notna().all()

assert (v3e1_df["sender_distinct_receivers_24h"] >= 0).all()

print("✅ Distinct-receiver counts are valid.")

✅ Distinct-receiver counts are valid.


In [311]:
v3e1_df["sender_distinct_receivers_24h"].describe(
    percentiles=[
        0.50,
        0.75,
        0.90,
        0.95,
        0.99,
        0.999,
    ]
)

count    3.731672e+06
mean     3.433236e+02
std      1.491836e+03
min      0.000000e+00
50%      2.000000e+00
75%      3.000000e+00
90%      8.000000e+00
95%      1.635000e+03
99%      8.865000e+03
99.9%    1.145400e+04
max      1.253800e+04
Name: sender_distinct_receivers_24h, dtype: float64

In [312]:
print(
    "No receiver activity in previous 24h:",
    (v3e1_df["sender_distinct_receivers_24h"] == 0).sum(),
)

print(
    "Maximum distinct receivers in previous 24h:",
    v3e1_df["sender_distinct_receivers_24h"].max(),
)

No receiver activity in previous 24h: 782174
Maximum distinct receivers in previous 24h: 12538


In [313]:
v3e1_df["sender_distinct_receivers_24h"].value_counts().head(20)

sender_distinct_receivers_24h
1      997700
0      782174
2      690681
3      399180
4      229779
5      132213
6       75891
7       43471
8       25564
9       14791
10       8485
11       4709
12       2822
13       1640
14        981
15        540
16        465
559       315
17        309
562       277
Name: count, dtype: int64

In [314]:
v3e1_df[
    [
        "transaction_timestamp",
        "from_bank_id",
        "from_account_id",
        "to_bank_id",
        "to_account_id",
        "sender_tx_count_1h",
        "sender_distinct_receivers_24h",
        "is_laundering",
    ]
].sort_values(
    "sender_distinct_receivers_24h",
    ascending=False,
).head(30)

,transaction_timestamp,from_bank_id,from_account_id,to_bank_id,to_account_id,sender_tx_count_1h,sender_distinct_receivers_24h,is_laundering
1869822,2022-09-03 00:00:00,70,100428660,3051,8016667A0,1033,12538,0
1869341,2022-09-02 23:59:00,70,100428660,11157,801A74F10,1048,12538,0
1869363,2022-09-02 23:59:00,70,100428660,1292,8005F0950,1048,12538,0
1869551,2022-09-03 00:00:00,70,100428660,11974,8023F82D0,1033,12538,0
1869579,2022-09-03 00:00:00,70,100428660,1362,8001C4B70,1033,12538,0
1869717,2022-09-03 00:00:00,70,100428660,795,80056F0D0,1033,12538,0
1869798,2022-09-03 00:00:00,70,100428660,8623,80CD0BEA0,1033,12538,0
1869097,2022-09-02 23:59:00,70,100428660,227171,80B0A4AB0,1048,12538,0
1869082,2022-09-02 23:59:00,70,100428660,795,802EF2C80,1048,12538,0
1868988,2022-09-02 23:59:00,70,100428660,8798,8034F4200,1048,12538,0


## V3E1

In [315]:
V3E1_NUMERIC_FEATURES = V3D2_NUMERIC_FEATURES + ["sender_distinct_receivers_24h"]

V3E1_CATEGORICAL_FEATURES = V3D2_CATEGORICAL_FEATURES.copy()

FEATURE_COLUMNS_V3E1 = V3E1_NUMERIC_FEATURES + V3E1_CATEGORICAL_FEATURES

In [316]:
print(
    "Added:",
    set(FEATURE_COLUMNS_V3E1) - set(FEATURE_COLUMNS_V3D2),
)

print(
    "Removed:",
    set(FEATURE_COLUMNS_V3D2) - set(FEATURE_COLUMNS_V3E1),
)

Added: {'sender_distinct_receivers_24h'}
Removed: set()


In [317]:
assert set(FEATURE_COLUMNS_V3E1) - set(FEATURE_COLUMNS_V3D2) == {
    "sender_distinct_receivers_24h"
}

assert set(FEATURE_COLUMNS_V3D2) - set(FEATURE_COLUMNS_V3E1) == set()

## Preprocessor

In [318]:
V3E1_NUMERIC_FEATURES
V3E1_CATEGORICAL_FEATURES

['receiving_currency', 'payment_currency', 'payment_format']

In [319]:
splits = build_walkforward_splits(v3e1_df)

In [320]:
for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(
        fold.name,
        f"train={len(train_idx):,}",
        f"validation={len(val_idx):,}",
    )


fold_1 train=2,076,752 validation=207,430
fold_2 train=2,284,182 validation=482,650
fold_3 train=2,766,832 validation=964,840


In [321]:
v3e1_fold_results = []
v3e1_runtime = []

for fold, (train_idx, val_idx) in zip(
    FROZEN_FOLDS,
    splits,
    strict=True,
):
    print(f"\n{'=' * 70}")
    print(f"Running {fold.name}")
    print(f"{'=' * 70}")

    X_train = v3e1_df.iloc[train_idx][FEATURE_COLUMNS_V3E1]
    y_train = v3e1_df.iloc[train_idx][TARGET_COL].to_numpy()

    X_val = v3e1_df.iloc[val_idx][FEATURE_COLUMNS_V3E1]
    y_val = v3e1_df.iloc[val_idx][TARGET_COL].to_numpy()

    print(f"Train: {len(y_train):,} rows | {y_train.sum():,} fraud")

    print(f"Validation: {len(y_val):,} rows | {y_val.sum():,} fraud")

    train_weights = compute_sample_weight(
        class_weight="balanced",
        y=y_train,
    )

    pipeline = make_lightgbm_pipeline(
        make_preprocessor, V3E1_NUMERIC_FEATURES, V3E1_CATEGORICAL_FEATURES
    )

    start = perf_counter()

    pipeline.fit(
        X_train,
        y_train,
        model__sample_weight=train_weights,
    )

    fit_seconds = perf_counter() - start

    start = perf_counter()

    val_scores = pipeline.predict_proba(X_val)[:, 1]

    predict_seconds = perf_counter() - start

    metrics = evaluate_fold_scores(
        fold_name=fold.name,
        y_true=y_val,
        y_score=val_scores,
        min_recall=0.80,
    )

    v3e1_fold_results.append(metrics)

    v3e1_runtime.append(
        {
            "fold": fold.name,
            "fit_seconds": fit_seconds,
            "predict_seconds": predict_seconds,
        }
    )

    print(f"\nAP        : {metrics['average_precision']:.6f}")
    print(f"PR lift   : {metrics['pr_lift']:.2f}x")
    print(f"ROC-AUC   : {metrics['roc_auc']:.6f}")
    print(f"Recall    : {metrics['recall_at_recall_floor']:.4%}")
    print(f"Precision : {metrics['precision_at_recall_floor']:.4%}")
    print(f"FP        : {metrics['false_positives_at_recall_floor']:,}")
    print(f"Alerts    : {metrics['alerts_at_recall_floor']:,}")
    print(f"Alert rate: {metrics['alert_rate_at_recall_floor']:.4%}")
    print(f"Fit time  : {fit_seconds:.2f}s")

    del (
        pipeline,
        X_train,
        X_val,
        y_train,
        y_val,
        train_weights,
        val_scores,
    )

    gc.collect()


Running fold_1
Train: 2,076,752 rows | 1,121 fraud
Validation: 207,430 rows | 407 fraud

AP        : 0.377397
PR lift   : 192.34x
ROC-AUC   : 0.983886
Recall    : 80.0983%
Precision : 6.5044%
FP        : 4,686
Alerts    : 5,012
Alert rate: 2.4162%
Fit time  : 24.96s

Running fold_2
Train: 2,284,182 rows | 1,528 fraud
Validation: 482,650 rows | 471 fraud

AP        : 0.302327
PR lift   : 309.80x
ROC-AUC   : 0.976807
Recall    : 80.0425%
Precision : 1.5986%
FP        : 23,206
Alerts    : 23,583
Alert rate: 4.8861%
Fit time  : 28.15s

Running fold_3
Train: 2,766,832 rows | 1,999 fraud
Validation: 964,840 rows | 1,028 fraud

AP        : 0.402554
PR lift   : 377.82x
ROC-AUC   : 0.983962
Recall    : 80.0584%
Precision : 6.0333%
FP        : 12,818
Alerts    : 13,641
Alert rate: 1.4138%
Fit time  : 33.86s


In [322]:
v3e1_results = build_fold_metrics_table(v3e1_fold_results)

v3e1_results[
    [
        "fold",
        "average_precision",
        "pr_lift",
        "roc_auc",
        "precision_at_recall_floor",
        "recall_at_recall_floor",
        "false_positives_at_recall_floor",
        "alerts_at_recall_floor",
        "alert_rate_at_recall_floor",
    ]
]

,fold,average_precision,pr_lift,roc_auc,precision_at_recall_floor,recall_at_recall_floor,false_positives_at_recall_floor,alerts_at_recall_floor,alert_rate_at_recall_floor
0,fold_1,0.377397,192.342899,0.983886,0.065044,0.800983,4686,5012,0.024162
1,fold_2,0.302327,309.804799,0.976807,0.015986,0.800425,23206,23583,0.048861
2,fold_3,0.402554,377.820990,0.983962,0.060333,0.800584,12818,13641,0.014138


In [323]:
comparison_v3e1 = v3d2_results.merge(
    v3e1_results[
        [
            "fold",
            "average_precision",
            "pr_lift",
            "roc_auc",
            "precision_at_recall_floor",
            "false_positives_at_recall_floor",
            "alert_rate_at_recall_floor",
        ]
    ],
    on="fold",
    suffixes=(
        "_v3d2",
        "_v3e1",
    ),
)

In [324]:
comparison_v3e1["ap_change"] = (
    comparison_v3e1["average_precision_v3e1"]
    - comparison_v3e1["average_precision_v3d2"]
)

comparison_v3e1["ap_change_pct"] = (
    comparison_v3e1["ap_change"] / comparison_v3e1["average_precision_v3d2"] * 100
)

comparison_v3e1["fp_change"] = (
    comparison_v3e1["false_positives_at_recall_floor_v3e1"]
    - comparison_v3e1["false_positives_at_recall_floor_v3d2"]
)

comparison_v3e1["alert_rate_change"] = (
    comparison_v3e1["alert_rate_at_recall_floor_v3e1"]
    - comparison_v3e1["alert_rate_at_recall_floor_v3d2"]
)

In [325]:
comparison_v3e1[
    [
        "fold",
        "average_precision_v3d2",
        "average_precision_v3e1",
        "ap_change",
        "ap_change_pct",
        "precision_at_recall_floor_v3d2",
        "precision_at_recall_floor_v3e1",
        "false_positives_at_recall_floor_v3d2",
        "false_positives_at_recall_floor_v3e1",
        "fp_change",
        "alert_rate_at_recall_floor_v3d2",
        "alert_rate_at_recall_floor_v3e1",
        "alert_rate_change",
    ]
]

,fold,average_precision_v3d2,average_precision_v3e1,ap_change,ap_change_pct,precision_at_recall_floor_v3d2,precision_at_recall_floor_v3e1,false_positives_at_recall_floor_v3d2,false_positives_at_recall_floor_v3e1,fp_change,alert_rate_at_recall_floor_v3d2,alert_rate_at_recall_floor_v3e1,alert_rate_change
0,fold_1,0.373212,0.377397,0.004185,1.121433,0.062464,0.065044,4893,4686,-207,0.025160,0.024162,-0.000998
1,fold_2,0.300583,0.302327,0.001744,0.580112,0.014368,0.015986,25861,23206,-2655,0.054362,0.048861,-0.005501
2,fold_3,0.393528,0.402554,0.009025,2.293435,0.047696,0.060333,16432,12818,-3614,0.017884,0.014138,-0.003746


## V3-E1 Decision — KEEP

**Feature tested:** V3-D2 + `sender_distinct_receivers_24h`

Adding the number of distinct receivers used by the sender during the previous 24 hours produced a modest but consistent improvement in ranking performance and a stronger improvement in operational alert quality.

Average Precision improved across all three chronological validation folds:

* Fold 1: **+1.12%**
* Fold 2: **+0.58%**
* Fold 3: **+2.29%**

Although the AP gains are smaller than those produced by some earlier behavioral features, the improvement is directionally consistent across every validation period.

Operational performance improved more substantially at the ≥80% recall operating point.

False positives decreased by:

* Fold 1: **−207**
* Fold 2: **−2,655**
* Fold 3: **−3,614**

Precision increased and alert rate decreased across all three folds. Fold 3 showed an especially meaningful improvement, with precision increasing from **4.77% to 6.03%** while false positives decreased from **16,432 to 12,818**.

The result suggests that recent counterparty dispersion contains useful information beyond transaction velocity and sender-receiver relationship history. Two senders can have similar transaction activity while exhibiting very different patterns in how broadly that activity is distributed across receivers.

**Decision:** KEEP `sender_distinct_receivers_24h`.

V3-E1 becomes the final selected feature set for the planned feature-engineering stage.


# Freeze the final experiment

In [326]:
DEV_END = pd.Timestamp("2022-09-08")

In [328]:
v3e1_df[v3e1_df["transaction_timestamp"] >= DEV_END]

,transaction_id,transaction_timestamp,from_bank_id,from_account_id,to_bank_id,to_account_id,amount_received,receiving_currency,amount_paid,payment_currency,...,sender_tx_count_1h,sender_amount_paid_sum_24h_same_currency,log_sender_amount_paid_sum_24h_same_currency,sender_hours_since_prev_tx,log_sender_hours_since_prev_tx,sender_historical_median_amount_paid,log_amount_paid_vs_sender_median,receiver_seen_before,sender_receiver_prior_tx_count,sender_distinct_receivers_24h


In [329]:
v3e1_dev_df = v3e1_df[v3e1_df["transaction_timestamp"] < DEV_END].copy()

print(v3e1_dev_df["transaction_timestamp"].min())
print(v3e1_dev_df["transaction_timestamp"].max())
print(v3e1_dev_df.shape)

2022-09-01 00:00:00
2022-09-07 23:59:00
(3731672, 30)


In [330]:
v3e1_dev_df.to_parquet(
    "../data/processed/v3e1_dev_sep1_7.parquet",
    index=False,
)

In [331]:
print("Numeric features:")
for feature in V3E1_NUMERIC_FEATURES:
    print(" -", feature)

print("\nCategorical features:")
for feature in V3E1_CATEGORICAL_FEATURES:
    print(" -", feature)

print("\nTotal model features:", len(FEATURE_COLUMNS_V3E1))

Numeric features:
 - hour_of_day
 - day_of_week
 - is_weekend
 - same_currency_flag
 - same_bank_flag
 - log_amount_received
 - log_amount_paid
 - sender_tx_count_1h
 - log_sender_amount_paid_sum_24h_same_currency
 - log_sender_hours_since_prev_tx
 - log_amount_paid_vs_sender_median
 - receiver_seen_before
 - sender_receiver_prior_tx_count
 - sender_distinct_receivers_24h

Categorical features:
 - receiving_currency
 - payment_currency
 - payment_format

Total model features: 17


In [332]:
v3e1_dev_df.columns

Index(['transaction_id', 'transaction_timestamp', 'from_bank_id',
       'from_account_id', 'to_bank_id', 'to_account_id', 'amount_received',
       'receiving_currency', 'amount_paid', 'payment_currency',
       'payment_format', 'is_laundering', 'transaction_date', 'hour_of_day',
       'day_of_week', 'is_weekend', 'same_currency_flag', 'same_bank_flag',
       'log_amount_received', 'log_amount_paid', 'sender_tx_count_1h',
       'sender_amount_paid_sum_24h_same_currency',
       'log_sender_amount_paid_sum_24h_same_currency',
       'sender_hours_since_prev_tx', 'log_sender_hours_since_prev_tx',
       'sender_historical_median_amount_paid',
       'log_amount_paid_vs_sender_median', 'receiver_seen_before',
       'sender_receiver_prior_tx_count', 'sender_distinct_receivers_24h'],
      dtype='str')

In [333]:
v3e1_results.to_csv(
    "../data/processed/v3e1_untuned_fold_results.csv",
    index=False,
)

# End